## Cell 1

In [1]:
import subprocess, time, os, shutil, glob

GDRIVE_CACHE = "/content/drive/MyDrive/ollama_cache"

from google.colab import drive
drive.mount("/content/drive")

def _dir_has_files(path):
    """Cek apakah folder ada DAN punya file di dalamnya."""
    return os.path.isdir(path) and len(os.listdir(path)) > 0

restored = False
if _dir_has_files(f"{GDRIVE_CACHE}/ollama") and _dir_has_files(f"{GDRIVE_CACHE}/models"):
    print(">>> Restoring Ollama from Google Drive cache...")
    try:
        os.makedirs("/usr/local/lib/ollama", exist_ok=True)
        !cp -r {GDRIVE_CACHE}/ollama/* /usr/local/lib/ollama/
        !ln -sf /usr/local/lib/ollama/ollama /usr/local/bin/ollama
        os.makedirs("/usr/share/ollama/.ollama/models", exist_ok=True)
        !cp -r {GDRIVE_CACHE}/models/* /usr/share/ollama/.ollama/models/
        # Verifikasi binary ada
        if shutil.which("ollama"):
            restored = True
            print(">>> Restored from cache!")
        else:
            print(">>> Restore gagal: ollama binary tidak ditemukan, install ulang...")
    except Exception as e:
        print(f">>> Restore error: {e}, install ulang...")
else:
    print(f">>> Cache tidak lengkap atau kosong di {GDRIVE_CACHE}")

# ini kalau tidak ada cache ollama, maka logika not restored dianggap mendownload ollama
if not restored:
    print(">>> Installing Ollama + downloading models...")
    !apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/tmp/ollama.log", "w"),
        stderr=open("/tmp/ollama_err.log", "w"),
    )
    time.sleep(5)

    !ollama pull deepseek-r1:7b
    !ollama pull llama-guard3:8b

    print(">>> Saving to Google Drive for next time...")
    # Bersihkan cache lama yang mungkin corrupt
    if os.path.exists(GDRIVE_CACHE):
        shutil.rmtree(GDRIVE_CACHE)
    os.makedirs(f"{GDRIVE_CACHE}/ollama", exist_ok=True)
    os.makedirs(f"{GDRIVE_CACHE}/models", exist_ok=True)
    !cp -r /usr/local/lib/ollama/* {GDRIVE_CACHE}/ollama/
    !cp -r /usr/share/ollama/.ollama/models/* {GDRIVE_CACHE}/models/
    print(">>> Saved to Google Drive!")

# Start ollama serve (idempotent — kalau sudah jalan, akan gagal diam-diam)
subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=open("/tmp/ollama_err.log", "w"),
)
time.sleep(5)

!ollama list

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> Cache tidak lengkap atau kosong di /content/drive/MyDrive/ollama_cache
>>> Installing Ollama + downloading models...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


>>> Saving to Google Drive for next time...
cp: cannot stat '/usr/share/ollama/.ollama/models/*': No such file

## Cell 2

In [2]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ind poppler-utils > /dev/null 2>&1

!pip install -q \
    fastapi \
    "uvicorn[standard]" \
    python-multipart \
    pydantic \
    pyyaml \
    loguru \
    tenacity \
    numpy \
    scikit-learn \
    langchain \
    langchain-community \
    langchain-core \
    langchain-experimental \
    langchain-huggingface \
    langchain-ollama \
    "chromadb>=0.5.4" \
    sentence-transformers \
    huggingface-hub \
    "unstructured[pdf]" \
    pdf2image \
    pytesseract \
    opencv-python-headless \
    Pillow \
    pdfminer.six \
    pypdf \
    pdfplumber \
    PyPDF2 \
    python-dotenv \
    requests \
    "ragas>=0.1.21" \
    "datasets>=2.14,<3" \
    pandas \
    pyngrok

print("done")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
done


## Cell 3

In [3]:
import os

# bakal memanggil logs, temp, evaluations, vector_db_lcel
WORKDIR = "/content/skripsi_backend"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

for d in ["logs", "temp", "evaluations", "vector_db_lcel"]:
    os.makedirs(d, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
!ls -la

Working directory: /content/skripsi_backend
total 36
drwxr-xr-x 7 root root 4096 May 24 16:23 .
drwxr-xr-x 1 root root 4096 May 24 15:46 ..
-rw-r--r-- 1 root root  383 May 24 16:23 book_registry.json
-rw-r--r-- 1 root root 1230 May 24 15:48 config.yaml
drwxr-xr-x 2 root root 4096 May 24 15:52 evaluations
drwxr-xr-x 2 root root 4096 May 24 15:47 logs
drwxr-xr-x 2 root root 4096 May 24 16:23 temp
drwxr-xr-x 3 root root 4096 May 24 15:50 vector_db_books
drwxr-xr-x 2 root root 4096 May 24 15:52 vector_db_lcel


## Cell 4

In [4]:



# %%writefile ingest.py


"""
Asisten Belajar SD FastAPI Backend dengan RAG Lokal (Ollama)
Versi: 7.2
"""


import os
import sys
import gc
import time
import shutil
import tempfile
import re
import json
import yaml
import random
import secrets
import threading
import functools
import platform
import asyncio
from contextlib import asynccontextmanager
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple, Any
from collections import defaultdict


# ============================================================
# PYDANTIC — CRITICAL IMPORT
# ============================================================
try:
    from pydantic import BaseModel, Field
    import pydantic as _pydantic
    _PYDANTIC_V2 = int(_pydantic.VERSION.split(".")[0]) >= 2
    print(f"… Pydantic {_pydantic.VERSION} loaded (V2={_PYDANTIC_V2})")
except ImportError as e:
    print(f"❌ FATAL: Pydantic not installed: {e}")
    print("   Run: pip install pydantic")
    sys.exit(1)


# ============================================================
# PYTORCH — LAZY LOADED IN detect_cuda_device()
# ============================================================
# Note: torch is NOT imported here. See detect_cuda_device() for lazy loading.


# FastAPI
try:
    from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks, Depends, Header, Request
    from fastapi.concurrency import run_in_threadpool
    from fastapi.middleware.cors import CORSMiddleware
    print("… FastAPI loaded")
except ImportError as e:
    print(f"❌ FATAL: FastAPI not installed: {e}")
    print("   Run: pip install fastapi uvicorn")
    sys.exit(1)




import pydantic as _pydantic
_PYDANTIC_V2 = int(_pydantic.VERSION.split(".")[0]) >= 2


if _PYDANTIC_V2:
    from pydantic import field_validator
else:
    from pydantic import validator  # type: ignore[no-redef]


# Logging
try:
    from loguru import logger
    print("… Loguru loaded")
except ImportError as e:
    print(f"⚠️  Loguru not installed: {e}. Using print() instead.")
    class LoggerStub:
        def info(self, msg): print(f"[INFO] {msg}")
        def warning(self, msg): print(f"[WARN] {msg}")
        def error(self, msg, **kw): print(f"[ERROR] {msg}")
        def debug(self, msg): print(f"[DEBUG] {msg}")
        def add(self, *a, **kw): pass
    logger = LoggerStub()
    sys.exit(1)  # Force exit - loguru is required


# Utilities
try:
    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_not_exception_type
    import numpy as np
    from sklearn.metrics.pairwise import cosine_similarity
    print("… Tenacity, NumPy, scikit-learn loaded")
except ImportError as e:
    print(f"❌ FATAL: Required utility library not installed: {e}")
    sys.exit(1)


# LangChain — CRITICAL
try:
    from langchain_community.document_loaders import UnstructuredPDFLoader
    from langchain_experimental.text_splitter import SemanticChunker
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_ollama import OllamaLLM
    from langchain_community.vectorstores import Chroma
    from langchain_community.vectorstores.utils import filter_complex_metadata
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
    from langchain_core.runnables import (
        RunnablePassthrough, RunnableLambda, RunnableParallel
    )
    from langchain_core.documents import Document
    print("… LangChain loaded")
except ImportError as e:
    print(f"❌ FATAL: LangChain not installed: {e}")
    print("   Run: pip install langchain langchain-community langchain-experimental langchain-huggingface langchain-ollama")
    sys.exit(1)


# Cross-encoder
try:
    from sentence_transformers import CrossEncoder
    print("… Sentence-transformers loaded")
except ImportError as e:
    print(f"❌ FATAL: sentence-transformers not installed: {e}")
    print("   Run: pip install sentence-transformers")
    sys.exit(1)


# OCR
try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
    print("… PyTesseract loaded")
except ImportError as e:
    PYTESSERACT_AVAILABLE = False
    print(f"⚠️  PyTesseract not installed: {e}. PDF OCR will be unavailable.")


try:
    from pdf2image import convert_from_path
    PDF2IMAGE_AVAILABLE = True
    print("… pdf2image loaded")
except ImportError:
    PDF2IMAGE_AVAILABLE = False
    print("⚠️ pdf2image not installed. OCR fallback unavailable.")


try:
    import cv2
    CV2_AVAILABLE = True
    print("… OpenCV loaded")
except ImportError:
    CV2_AVAILABLE = False
    print("⚠️ OpenCV not installed. OCR preprocessing unavailable.")


# RAGAS — OPTIONAL (supports 0.1.x and 0.2.x)
# Compatibility shim: langchain-community >= 0.3 removed chat_models.vertexai
# which some ragas versions try to import at module level. Stub it out so that
# ragas can be imported successfully even without the vertexai package.
try:
    import langchain_community.chat_models.vertexai  # noqa: F401
except (ModuleNotFoundError, ImportError):
    import sys as _sys
    from types import ModuleType as _ModuleType
    _vertexai_stub = _ModuleType("langchain_community.chat_models.vertexai")
    # Stub any attribute ragas might reference (e.g. ChatVertexAI)
    _vertexai_stub.ChatVertexAI = None
    _sys.modules["langchain_community.chat_models.vertexai"] = _vertexai_stub

try:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    )
    # LangchainLLMWrapper dan LangchainEmbeddingsWrapper ada di path sama di 0.1.x dan 0.2.x
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_ollama import OllamaEmbeddings
    from datasets import Dataset
    import pandas as pd

    # Deteksi versi RAGAS
    import importlib.metadata as _imeta
    try:
        _ragas_ver = _imeta.version("ragas")
    except Exception:
        _ragas_ver = "unknown"

    RAGAS_AVAILABLE = True
    print(f"… RAGAS loaded successfully (v{_ragas_ver})")

except ImportError as e:
    RAGAS_AVAILABLE = False
    # Provide dummy types so the rest of the code can reference them without NameError
    class LangchainLLMWrapper: pass       # noqa: E701
    class LangchainEmbeddingsWrapper: pass # noqa: E701
    logger.warning(f"⚠️  RAGAS tidak tersedia: {e}. Evaluasi RAGAS tidak bisa dijalankan.")
    logger.warning("   Solusi: pastikan Cell 4 (pip install) sudah dijalankan ulang di Colab.")


if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())




def _force_rmtree(path: str, retries: int = 5, delay: float = 1.0):
    """shutil.rmtree with retry for Windows file locking."""
    for attempt in range(retries):
        try:
            shutil.rmtree(path)
            return
        except PermissionError:
            if attempt < retries - 1:
                gc.collect()
                time.sleep(delay)
            else:
                raise




def _release_chroma(vector_db):
    """Properly release ChromaDB client to free file locks on Windows."""
    if vector_db is None:
        return
    try:
        vector_db._client.clear_system_cache()
    except Exception:
        pass
    try:
        vector_db._client._server.stop()
    except Exception:
        pass
    try:
        del vector_db._client
    except Exception:
        pass
    gc.collect()
    # Give Windows time to release file handles
    if sys.platform == "win32":
        time.sleep(0.5)




# ========================================
# CUDA DETECTION
# ========================================
def detect_cuda_device():
    """Deteksi GPU NVIDIA dan set memory limit 80%."""
    try:
        import torch
    except ImportError:
        print("⚠️ PyTorch tidak terinstall. Pastikan Cell 2 sudah dijalankan!")
        print("   Using CPU fallback.")
        return None, "CPU", False


    try:
        if torch.cuda.is_available():
            gpu_count = torch.cuda.device_count()
            gpu_name = torch.cuda.get_device_name(0)
            cuda_version = torch.version.cuda
            # Set memory fraction 80% dari total VRAM
            torch.cuda.set_per_process_memory_fraction(0.8, device=0)
            print(f"… CUDA detected: {gpu_count} GPU(s)")
            print(f"   GPU Name       : {gpu_name}")
            print(f"   CUDA Version   : {cuda_version}")
            print(f"   Memory limit   : 80% of VRAM")
            return torch.device("cuda"), gpu_name, True
        else:
            print("⚠️ CUDA not available (no NVIDIA GPU), using CPU")
            return torch.device("cpu"), "CPU", False
    except Exception as e:
        print(f"⚠️ CUDA detection error: {e}. Falling back to CPU.")
        return torch.device("cpu"), "CPU", False


DEVICE, DEVICE_NAME, CUDA_AVAILABLE = detect_cuda_device() or (None, "CPU", False)


# ========================================
# HELPER: STRIP DEEPSEEK THINKING
# ========================================


def strip_thinking(text: str) -> str:
    """
    Hapus bagian <think>...</think> dari output DeepSeek-R1.
    Siswa tidak boleh melihat proses berpikir internal model.


    FIX: Jika setelah strip hasilnya kosong (model menaruh SELURUH jawaban
    di dalam <think>), ekstrak paragraf terakhir dari isi think sebagai jawaban.
    """
    original = text


    # Coba ekstrak isi think tag dulu (untuk fallback jika nanti kosong)
    think_content = ""
    think_match = re.search(r'<think>(.*?)</think>', text, flags=re.DOTALL)
    if think_match:
        think_content = think_match.group(1).strip()


    # Standard stripping
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<think>.*',          '', text, flags=re.DOTALL)
    text = re.sub(r'.*</think>',         '', text, flags=re.DOTALL)
    text = re.sub(r'(?i)^thinking:.*\n?','', text, flags=re.MULTILINE)
    result = text.strip()


    # FIX: Jika hasil kosong tapi ada konten di think block,
    # ambil paragraf terakhir yang substantif sebagai jawaban
    if len(result) < 10 and think_content:
        # Split by paragraphs dan ambil yang terakhir/terpanjang
        paragraphs = [p.strip() for p in think_content.split('\n\n') if p.strip()]
        if not paragraphs:
            paragraphs = [p.strip() for p in think_content.split('\n') if p.strip()]


        # Filter: ambil paragraf yang seperti jawaban (bukan proses berpikir)
        # Heuristik: paragraf terakhir biasanya jawaban final model
        candidates = []
        for p in paragraphs:
            # Skip baris yang jelas proses berpikir
            if re.match(r'^(Let me|I need|I should|I\'ll|Hmm|Ok|Wait|The question)', p, re.IGNORECASE):
                continue
            if len(p) > 20:  # minimal substantif
                candidates.append(p)


        if candidates:
            # Ambil paragraf terakhir (biasanya jawaban final)
            result = candidates[-1]
            # Bersihkan prefix yang sering muncul di think
            result = re.sub(r'^(Jadi,?\s*|Kesimpulan:?\s*|Jawaban:?\s*)', '', result, flags=re.IGNORECASE)
            result = result.strip()


    return result




def clean_json_output(text: str) -> str:
    """Strip thinking lalu bersihkan markdown code block."""
    text = strip_thinking(text)
    text = re.sub(r'```(?:json)?\s*', '', text)
    text = re.sub(r'```', '', text)
    return text.strip()




# ========================================
# PYDANTIC MODELS
# ========================================


class QuizQuestion(BaseModel):
    soal: str       = Field(..., description="Pertanyaan quiz")
    opsi: List[str] = Field(..., description="4 pilihan jawaban (A, B, C, D)")
    kunci: str      = Field(..., description="Jawaban yang benar (A/B/C/D)")
    penjelasan: str = Field(..., description="Penjelasan jawaban")


    if _PYDANTIC_V2:
        @field_validator('opsi', mode='before')
        @classmethod
        def validate_opsi(cls, v):
            if len(v) != 4:
                raise ValueError("Harus ada 4 pilihan jawaban")
            return v


        @field_validator('kunci', mode='before')
        @classmethod
        def validate_kunci(cls, v):
            v = v.strip().upper()
            if v not in ['A', 'B', 'C', 'D']:
                raise ValueError("Kunci harus A, B, C, atau D")
            return v
    else:
        @validator('opsi')  # type: ignore[no-redef]
        @classmethod
        def validate_opsi(cls, v):  # type: ignore[no-redef]
            if len(v) != 4:
                raise ValueError("Harus ada 4 pilihan jawaban")
            return v


        @validator('kunci')  # type: ignore[no-redef]
        @classmethod
        def validate_kunci(cls, v):  # type: ignore[no-redef]
            v = v.strip().upper()
            if v not in ['A', 'B', 'C', 'D']:
                raise ValueError("Kunci harus A, B, C, atau D")
            return v




class QuizResponse(BaseModel):
    questions: List[QuizQuestion]




class ChatRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=500)
    include_sources:    bool = True
    include_validation: bool = True
    run_ragas:          bool = False
    ground_truth: Optional[str] = None


    if _PYDANTIC_V2:
        @field_validator('prompt', mode='before')
        @classmethod
        def validate_prompt(cls, v):
            if not v.strip():
                raise ValueError("Prompt cannot be empty")
            return v.strip()
    else:
        @validator('prompt')  # type: ignore[no-redef]
        @classmethod
        def validate_prompt(cls, v):  # type: ignore[no-redef]
            if not v.strip():
                raise ValueError("Prompt cannot be empty")
            return v.strip()




class RAGASEvaluationRequest(BaseModel):
    questions:     List[str]           = Field(..., description="List pertanyaan")
    ground_truths: Optional[List[str]] = Field(None, description="Ground truth (opsional)")




class AuthVerifyRequest(BaseModel):
    token: str = Field(..., min_length=1, max_length=256)
    role: str  = Field(..., description="Role login: guru atau siswa")


    if _PYDANTIC_V2:
        @field_validator('token', mode='before')
        @classmethod
        def validate_token(cls, v):
            token = str(v).strip()
            if not token:
                raise ValueError("Token tidak boleh kosong")
            return token


        @field_validator('role', mode='before')
        @classmethod
        def validate_role(cls, v):
            role = str(v).strip().lower()
            if role not in ("guru", "siswa"):
                raise ValueError("Role harus 'guru' atau 'siswa'")
            return role
    else:
        @validator('token')  # type: ignore[no-redef]
        @classmethod
        def validate_token(cls, v):  # type: ignore[no-redef]
            token = str(v).strip()
            if not token:
                raise ValueError("Token tidak boleh kosong")
            return token


        @validator('role')  # type: ignore[no-redef]
        @classmethod
        def validate_role(cls, v):  # type: ignore[no-redef]
            role = str(v).strip().lower()
            if role not in ("guru", "siswa"):
                raise ValueError("Role harus 'guru' atau 'siswa'")
            return role




# ========================================
# RATE LIMITER (in-memory, tanpa dependensi eksternal)
# ========================================


class SimpleRateLimiter:
    """
    Rate limiter berbasis sliding window.
    Default: 30 request per menit per token/IP.
    Siswa dan guru memiliki bucket terpisah sesuai token mereka.
    """
    import time as _time


    def __init__(self, max_requests: int = 30, window_seconds: int = 60):
        self.max_requests   = max_requests
        self.window         = window_seconds
        self._buckets: Dict[str, list] = defaultdict(list)
        self._lock          = threading.Lock()


    def is_allowed(self, key: str) -> bool:
        import time
        now = time.monotonic()
        with self._lock:
            bucket = self._buckets[key]
            # Buang request lama di luar window
            self._buckets[key] = [t for t in bucket if now - t < self.window]
            if len(self._buckets[key]) >= self.max_requests:
                return False
            self._buckets[key].append(now)
            return True




rate_limiter = SimpleRateLimiter(max_requests=30, window_seconds=60)




# ========================================
# CONFIGURATION
# ========================================


class Config:
    def __init__(self, config_path: str = "config.yaml"):
        self.config_path = config_path
        self.config = self._load_config()
        self._setup_paths()
        self._setup_external_tools()


    def _load_config(self) -> dict:
        try:
            if os.path.exists(self.config_path):
                with open(self.config_path, 'r', encoding='utf-8') as f:
                    return yaml.safe_load(f)
            logger.warning(f"Config tidak ditemukan: {self.config_path}. Pakai default.")
            return self._get_default_config()
        except Exception as e:
            logger.error(f"Error load config: {e}. Pakai default.")
            return self._get_default_config()


    def _get_default_config(self) -> dict:
        return {
            "paths": {
                "vector_db":   "./vector_db_lcel",
                "evaluations": "./evaluations",
                "logs":        "./logs",
                "temp":        "./temp"
            },
            "llm": {
                "primary_model":      "deepseek-r1:7b",
                "llama_guard_model":  "llama-guard3:8b",
                "ollama_base_url":    "http://localhost:11434",
                "temperature":        {"qa": 0.1, "quiz": 0.3, "guard": 0.0},
                "timeout":            180,
                "guard_timeout":      30
            },
            "embedding": {
                "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                "device":     "cpu"
            },
            "pdf": {
                "strategy":       "hi_res",
                "extract_images": True,
                "infer_tables":   True,
                "languages":      ["ind", "eng"],
                "ocr_languages":  "ind+eng",
                "ocr_fallback":   True,
                "tesseract_available": False,
                "ocr_min_chars_per_page": 50,
                "max_size_mb":    50          # batas upload PDF dalam MB
            },
            "retrieval": {
                "method":      "mmr",
                "top_k":       8,
                "fetch_k":     30,
                "lambda_mult": 0.8,
                "rerank":      {"enabled": True, "top_n": 4}
            },
            "relevance": {
                "threshold": 0.6,
                "methods":   ["cosine_similarity", "semantic_overlap"]
            },
            "quiz": {
                "default_questions":   3,
                "max_questions":       10,
                "chunks_per_question": 3,
                "max_retry":           3
            },
            "anti_hallucination": {
                "strict_mode":               True,
                "overlap_refuse_threshold":  0.30,
                "min_context_overlap":       0.30
            },
            "ragas": {
                "enabled":     True,
                "result_file": "./evaluations/ragas_results.jsonl"
            },
            "api": {
                "host":         "0.0.0.0",
                "port":         8000,
                "cors_origins": ["http://localhost:3000", "http://localhost:5173"]
            },


            "auth": {
                "guru_token":  os.environ.get("GURU_TOKEN",  "GANTI-GURU-TOKEN-PRODUCTION"),
                "siswa_token": os.environ.get("SISWA_TOKEN", "GANTI-SISWA-TOKEN-PRODUCTION")
            }
        }


    def _setup_paths(self):
        for _, path_value in self.config.get("paths", {}).items():
            os.makedirs(path_value, exist_ok=True)


    def _setup_external_tools(self):
        system = platform.system()
        tesseract_available = False
        # FIX: Hapus path developer spesifik (C:\Users\cocyo\...).
        # Tambahkan path standar saja; user bisa override via config.yaml.
        tesseract_paths = {
            "Windows": [
                r"C:\Program Files\Tesseract-OCR\tesseract.exe",
                r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
                r"C:\tools\Tesseract-OCR\tesseract.exe"
            ],
            "Linux":  ["/usr/bin/tesseract", "/usr/local/bin/tesseract"],
            "Darwin": ["/usr/local/bin/tesseract", "/opt/homebrew/bin/tesseract"]
        }


        # Cek juga path custom dari config / env var
        custom_path = self.get("pdf.tesseract_path") or os.environ.get("TESSERACT_PATH")
        if custom_path:
            tesseract_paths.get(system, []).insert(0, custom_path)


        tesseract_cmd = None
        for path in tesseract_paths.get(system, []):
            if os.path.exists(path):
                tesseract_cmd = path
                break


        if tesseract_cmd and PYTESSERACT_AVAILABLE:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
            tesseract_dir = os.path.dirname(tesseract_cmd)
            if tesseract_dir not in os.environ.get('PATH', ''):
                os.environ['PATH'] = tesseract_dir + os.pathsep + os.environ.get('PATH', '')
            logger.info(f"… Tesseract: {tesseract_cmd}")
            tesseract_available = True
        elif tesseract_cmd and not PYTESSERACT_AVAILABLE:
            logger.warning("⚠️ Tesseract executable ditemukan, tapi pytesseract belum terinstall.")
        else:
            logger.warning("⚠️ Tesseract tidak ditemukan. Set TESSERACT_PATH atau pdf.tesseract_path di config.yaml.")


        # Simpan status agar pipeline ingest bisa memutuskan kapan OCR boleh dipakai.
        self.config.setdefault("pdf", {})["tesseract_available"] = tesseract_available


        if system == "Windows":
            poppler_candidates = [
                os.environ.get("POPPLER_PATH", ""),
                r"C:\Program Files\poppler-24.02.0\Library\bin",
                r"C:\Program Files\poppler\Library\bin",
                r"C:\poppler-24.02.0\Library\bin",
                r"C:\poppler\Library\bin",
                r"C:\tools\poppler\Library\bin",
                r"C:\tools\poppler-24.02.0\Library\bin",
            ]
            for poppler_path in poppler_candidates:
                if poppler_path and os.path.exists(poppler_path):
                    if poppler_path not in os.environ.get('PATH', ''):
                        os.environ['PATH'] += os.pathsep + poppler_path
                    logger.info(f"… Poppler: {poppler_path}")
                    break
            else:
                logger.warning(
                    "⚠️ Poppler tidak ditemukan di path standar Windows. "
                    "Set env var POPPLER_PATH=C:\\path\\to\\poppler\\Library\\bin "
                    "atau pdf.poppler_path di config.yaml."
                )


    def get(self, key: str, default=None):
        keys  = key.split('.')
        value = self.config
        for k in keys:
            if isinstance(value, dict):
                value = value.get(k, default)
            else:
                return default
        return value if value is not None else default




config = Config()


_cors_env = os.environ.get("CORS_ORIGINS")
if _cors_env:
    config.config.setdefault("api", {})["cors_origins"] = [
        o.strip() for o in _cors_env.split(",") if o.strip()
    ]


log_dir = config.get("paths.logs", "./logs")
logger.add(
    f"{log_dir}/assistant_{{time}}.log",
    rotation="10 MB", retention="30 days", level="INFO",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}"
)




# ========================================
# AUTENTIKASI BERBASIS PERAN
# ========================================
# Setiap request harus menyertakan header:
#   X-Auth-Token: <token>
#
# Token guru   → akses ke semua endpoint termasuk /upload
# Token siswa  → hanya /chat dan /quiz
#
# Contoh penggunaan (curl):
#   curl -H "X-Auth-Token: $GURU_TOKEN" -F "file=@buku.pdf" http://localhost:8000/upload
#   curl -H "X-Auth-Token: $SISWA_TOKEN" -X POST -d '{"prompt":"..."}' http://localhost:8000/chat


def get_current_role(x_auth_token: Optional[str] = Header(None)) -> str:
    # FIX: Prioritaskan env var, fallback ke config
    guru_token  = os.environ.get("GURU_TOKEN")  or config.get("auth.guru_token",  "")
    siswa_token = os.environ.get("SISWA_TOKEN") or config.get("auth.siswa_token", "")


    # Dev fallback saat config masih placeholder bawaan
    if not guru_token or "GANTI" in guru_token:
        guru_token = os.environ.get("DEV_GURU_TOKEN", "Guru2026")
    if not siswa_token or "GANTI" in siswa_token:
        siswa_token = os.environ.get("DEV_SISWA_TOKEN", "Siswa2026")


    if not x_auth_token:
        raise HTTPException(
            status_code=401,
            detail="Header X-Auth-Token wajib disertakan."
        )


    # FIX: Gunakan secrets.compare_digest() — cegah timing attack
    token_bytes = x_auth_token.encode()
    if guru_token  and secrets.compare_digest(token_bytes, guru_token.encode()):
        return "guru"
    if siswa_token and secrets.compare_digest(token_bytes, siswa_token.encode()):
        return "siswa"


    logger.warning(f"❌ Auth failed: invalid token (received {len(x_auth_token)} chars)")
    raise HTTPException(
        status_code=403,
        detail="Token tidak valid. Pastikan X-Auth-Token header tepat. Hubungi administrator."
    )




def require_guru(role: str = Depends(get_current_role)) -> str:
    """Dependency khusus endpoint guru — tolak jika bukan guru."""
    if role != "guru":
        raise HTTPException(
            status_code=403,
            detail="⛔ Hanya guru yang dapat mengakses fitur ini."
        )
    return role




def require_any(role: str = Depends(get_current_role)) -> str:
    """Dependency yang mengizinkan guru maupun siswa."""
    return role




# FIX: Rate limit dependency — batasi 30 request/menit per token
async def rate_limit(request: Request) -> None:
    key = request.headers.get("x-auth-token") or (
        request.client.host if request.client else "unknown"
    )
    if not rate_limiter.is_allowed(key):
        raise HTTPException(
            status_code=429,
            detail="Terlalu banyak request. Coba lagi dalam 1 menit."
        )




# ========================================
# LLAMA GUARD 3 — TOXICITY FILTER
# ========================================


class LlamaGuardFilter:
    """
    Modul Toxicity Filtering 2-Layer:
      Layer 1: Rule-Based (keyword + regex) — SELALU aktif, tanpa LLM
      Layer 2: LlamaGuard 3 8B + Safety Policy + Prompt (Bahasa Indonesia)


    10 Kategori keamanan (K1–K10):
      K1 – Konten Seksual Eksplisit
      K2 – Kekerasan Grafis dan Kekejaman
      K3 – Diskriminasi Rasial dan Etnis
      K4 – Penghinaan, Makian, dan Pelecehan
      K5 – Niat Berbahaya dan Instruksi Berbahaya
      K6 – Body Shaming dan Diskriminasi Penampilan Fisik
      K7 – Diskriminasi Gender dan Orientasi Seksual
      K8 – Diskriminasi Berdasarkan Lokasi dan Asal Daerah
      K9 – Konten NSFW (Tidak Pantas untuk Lingkungan Umum)
      K10 – Bahaya Bunuh Diri dan Menyakiti Diri Sendiri
    """


    EDU_MESSAGES: Dict[str, str] = {
        "K1": ("⚠️ Pertanyaan kamu mengandung konten yang tidak pantas. "
               "Fokus pada materi pelajaran ya! 😊"),
        "K2": ("⚠️ Pesan kamu mengandung unsur kekerasan. "
               "Mari kita jaga suasana belajar yang aman dan nyaman! 🤝"),
        "K3": ("⚠️ Pesan kamu mengandung diskriminasi. "
               "Kita harus saling menghormati semua orang ya! 🌈"),
        "K4": ("⚠️ Hei! Kata yang kamu tulis kurang sopan. "
               "Yuk gunakan bahasa yang baik dan santun ya! 😊"),
        "K5": ("⚠️ Pertanyaan kamu mengandung hal yang berbahaya. "
               "Tanyakan sesuatu yang berkaitan dengan materi pelajaran ya! 📚"),
        "K6": ("⚠️ Jangan mengolok-olok penampilan orang lain ya. "
               "Setiap orang itu unik dan berharga! ¨"),
        "K7": ("⚠️ Pesan kamu mengandung diskriminasi gender. "
               "Kita harus menghormati semua orang! 🤝"),
        "K8": ("⚠️ Jangan merendahkan orang berdasarkan daerah asalnya. "
               "Semua daerah di Indonesia itu istimewa! 🇮🇩"),
        "K9": ("⚠️ Pertanyaan kamu tidak pantas untuk lingkungan belajar. "
               "Gunakan bahasa yang sopan ya! 📖"),
        "K10": ("⚠️ Jika kamu merasa sedih atau butuh bantuan, "
                "ceritakan ke guru atau orang tua ya. Kamu tidak sendirian! 💙"),
    }
    DEFAULT_EDU_MSG = ("⚠️ Maaf, pertanyaan kamu tidak dapat diproses. "
                       "Gunakan bahasa yang sopan dan tanyakan seputar materi pelajaran ya! 😊")


    SAFETY_POLICY = """K1: Konten Seksual Eksplisit.
AI tidak boleh membuat atau memfasilitasi konten yang menggambarkan tindakan seksual eksplisit, pornografi, atau materi dewasa yang vulgar. Mencakup deskripsi aktivitas seksual, permintaan konten pornografi, atau fetish seksual.


K2: Kekerasan Grafis dan Kekejaman.
AI tidak boleh menghasilkan konten kekerasan ekstrem, penyiksaan, pembunuhan sadis, atau skenario kekerasan brutal yang dapat menyebabkan trauma atau mendorong tindakan kekerasan.


K3: Diskriminasi Rasial dan Etnis.
AI tidak boleh menghasilkan konten yang merendahkan atau mempromosikan prasangka terhadap kelompok ras, suku, atau etnis tertentu. Termasuk slur rasial, lelucon rasis, atau klaim superioritas ras.


K4: Penghinaan, Makian, dan Pelecehan.
AI tidak boleh menghasilkan konten yang menghina, melecehkan, atau menyerang martabat individu/kelompok dengan kata-kata kasar, makian, atau bahasa yang merendahkan.


K5: Niat Berbahaya dan Instruksi Berbahaya.
AI tidak boleh memberikan panduan atau instruksi yang dapat digunakan untuk menyebabkan kerugian fisik/material. Termasuk panduan membuat senjata/bom, instruksi kegiatan ilegal, cara mendapatkan narkoba, atau informasi yang memfasilitasi kejahatan.


K6: Body Shaming dan Diskriminasi Penampilan Fisik.
AI tidak boleh menghasilkan konten yang merendahkan atau mempermalukan seseorang berdasarkan penampilan fisik (berat badan, tinggi, warna kulit, fitur fisik lainnya).


K7: Diskriminasi Gender dan Orientasi Seksual.
AI tidak boleh menghasilkan konten yang mendiskriminasi berdasarkan jenis kelamin, identitas gender, atau orientasi seksual.


K8: Diskriminasi Berdasarkan Lokasi dan Asal Daerah.
AI tidak boleh menghasilkan konten yang merendahkan orang dari lokasi geografis, kota, daerah, atau negara tertentu.


K9: Konten NSFW (Tidak Pantas untuk Lingkungan Umum).
AI tidak boleh menghasilkan konten tidak pantas untuk lingkungan pendidikan, termasuk bahasa kasar berlebihan, konten vulgar, atau materi yang sangat menyinggung.


K10: Bahaya Bunuh Diri dan Menyakiti Diri Sendiri.
AI tidak boleh memberikan informasi, metode, atau dorongan yang dapat memfasilitasi bunuh diri atau menyakiti diri sendiri."""


    SYSTEM_PROMPT = (
        "Anda adalah sistem evaluasi keamanan konten cerdas yang dirancang khusus "
        "untuk konten berbahasa Indonesia dalam konteks pendidikan Sekolah Dasar "
        "(usia 6-12 tahun). Tugas Anda adalah menganalisis pesan pengguna dan "
        "menentukan apakah pesan tersebut mengandung konten berbahaya atau tidak aman.\n\n"
        "Pedoman analisis:\n"
        "1. Perhatikan konteks bahasa Indonesia secara menyeluruh, termasuk:\n"
        "   - Kata slang, gaul, dan ekspresi lokal yang bernuansa toxic\n"
        "   - Idiom daerah yang mungkin mengandung makna merendahkan\n"
        "   - Kalimat terselubung atau eufemisme untuk konten berbahaya\n"
        "2. Evaluasi niat di balik pesan, bukan hanya kata-kata harfiah\n"
        "3. Pertimbangkan dampak potensial terhadap anak usia SD\n"
        "4. Bersikaplah konsisten dan objektif dalam setiap penilaian\n"
        "5. Jangan abaikan konten yang tampak tidak berbahaya namun menyiratkan "
        "bahaya secara implisit"
    )


    # ══════════════════════════════════════════════════════════════
    # RULE-BASED: Keyword & Pattern Matching (Layer 1)
    # ══════════════════════════════════════════════════════════════


    TOXIC_KEYWORDS_CATEGORIZED: Dict[str, list] = {
        "K1": [
            "seks", "seksual", "pornografi", "porn", "bokep", "hentai",
            "bugil", "telanjang", "jorok", "cabul", "vulgar",
            "ngentot", "memek", "kontol", "vagina", "penis", "payudara",
            "masturbasi", "fetish", "orgasme", "ejakulasi", "onani",
            "hubungan badan", "bercinta", "ngewe", "esek", "mesum",
            "adegan dewasa", "video panas", "foto bugil", "konten dewasa",
            "porno", "ngeseks", "pepek", "jembut", "entot", "pentil",
        ],
        "K2": [
            "bunuh", "membunuh", "menyiksa", "siksa", "kekerasan",
            "melukai", "membantai", "bantai", "kejam", "brutal",
            "menyakiti", "penggal", "mutilasi", "sadis", "pembunuhan",
            "tikam", "bacok", "tembak", "ledakkan", "hancurkan",
            "hajar", "pukul sampai mati", "gorok",
        ],
        "K3": [
            "rasis", "rasisme", "diskriminasi", "primitif", "inferior",
            "kafir", "negro", "pribumi bodoh", "suku rendah", "ras rendah",
            "cina babi", "inlander",
        ],
        "K4": [
            "bangsat", "brengsek", "bajingan", "keparat", "kurang ajar",
            "sialan", "kampret", "bedebah", "laknat", "biadab",
            "goblok", "goblog", "dungu", "idiot", "tolol", "bodoh banget",
            "tai", "taik", "tahi", "asu", "jancok", "jancuk", "pukimak",
            "celaka", "setan", "iblis", "anjing", "anjg", "anjir",
            "babi", "kimak", "pantek", "lonte", "pelacur", "sundal",
            "bngst", "bgst",
        ],
        "K5": [
            "narkoba", "sabu", "meth", "heroin", "ekstasi", "putaw",
            "overdosis", "racun", "bom", "senjata", "pistol ilegal",
            "cara membuat bom", "cara meracuni",
        ],
        "K6": [
            "gendut", "gemuk jelek", "kurus kering", "ceking",
            "pesek", "jerawatan", "cacat fisik",
            "kulit hitam jelek", "pendek banget",
        ],
        "K7": [
            "banci", "bencong", "waria hina", "gay jijik",
            "lesbi kotor", "perempuan lemah", "cewek goblok",
        ],
        "K8": [
            "orang desa bodoh", "kampungan", "udik",
            "orang pelosok", "daerah terbelakang",
        ],
        "K9": [
            "xxx", "mabuk",
        ],
        "K10": [
            "bunuh diri", "self harm", "menyiksa diri",
            "cara mati", "cara bunuh diri",
        ],
    }


    TOXIC_PATTERNS_CATEGORIZED = [
        (r"cara\s+(paling\s+)?(mudah\s+)?membunuh", "K2"),
        (r"cara\s+(paling\s+)?melukai\s+orang", "K2"),
        (r"cara\s+menyakiti\s+(orang|seseorang)", "K2"),
        (r"bagaimana\s+(cara\s+)?membunuh", "K2"),
        (r"(buat|bikin)\s+bom", "K2"),
        (r"adegan\s+porno(grafi)?", "K1"),
        (r"deskripsi(kan)?\s+(adegan\s+)?seks", "K1"),
        (r"jelaskan\s+(secara\s+rinci\s+)?tentang\s+seks", "K1"),
        (r"ceritakan\s+(tentang\s+)?pengalaman\s+seksual", "K1"),
        (r"lelucon\s+jorok", "K1"),
        (r"posisi\s+seksual", "K1"),
        (r"fantasi\s+seksual", "K1"),
        (r"cara\s+(membuat|bikin)\s+racun", "K5"),
        (r"cara\s+dapat\s+narkoba", "K5"),
        (r"instruksi\s+(untuk\s+)?bunuh\s+diri", "K10"),
        (r"tutorial\s+(cara\s+)?mati", "K10"),
        (r'k[o0]+nt[o0]+l', "K1"),
        (r'm[e3]+m[e3]+k', "K1"),
        (r'[a4]nj[i1!]+ng', "K4"),
        (r'b[a4]+ng?s[a4]+t', "K4"),
        (r'g[o0]+bl[o0]+k', "K4"),
        (r't[o0]+l[o0]+l', "K4"),
        (r'b[a4]+b[i1!]', "K4"),
        (r'n[g9]+[e3]nt[o0]+t', "K1"),
        (r'n[g9]+[e3]w[e3]', "K1"),
        (r'p[e3]+l[a4]+c[u]+r', "K1"),
        (r'l[o0]+nt[e3]', "K1"),
    ]


    _FLAT_KEYWORDS: list = []


    @classmethod
    def _build_flat_keywords(cls):
        if not cls._FLAT_KEYWORDS:
            for cat, words in cls.TOXIC_KEYWORDS_CATEGORIZED.items():
                for w in words:
                    cls._FLAT_KEYWORDS.append((w.lower(), cat))


    def __init__(self):
        self._build_flat_keywords()
        guard_model = config.get("llm.llama_guard_model", "llama-guard3:8b")
        timeout     = config.get("llm.guard_timeout", 30)
        try:
            self.llm = OllamaLLM(
                model=guard_model,
                temperature=0.0,
                timeout=timeout,
                base_url=config.get("llm.ollama_base_url", "http://localhost:11434")
            )
            self.enabled = True
            logger.info(f"… LlamaGuardFilter siap — model: {guard_model}")
        except Exception as e:
            logger.warning(f"⚠️ LlamaGuardFilter gagal inisialisasi: {e}. "
                           "Fallback ke Rule-Based only.")
            self.llm     = None
            self.enabled = False


    def _build_prompt(self, user_text: str) -> str:
        return (
            f"<|begin_of_text|>[INST] {self.SYSTEM_PROMPT}\n\n"
            "Tugas: Periksa apakah ada konten tidak aman dalam pesan 'Pengguna' "
            "berdasarkan kebijakan keamanan kami dengan kategori berikut.\n\n"
            "<MULAI KATEGORI KONTEN TIDAK AMAN>\n"
            f"{self.SAFETY_POLICY.strip()}\n"
            "<AKHIR KATEGORI KONTEN TIDAK AMAN>\n\n"
            "<MULAI PERCAKAPAN>\n"
            f"Pengguna: {user_text}\n"
            "<AKHIR PERCAKAPAN>\n\n"
            "Berikan penilaian keamanan Anda untuk Pengguna dalam percakapan di atas:\n"
            "- Baris pertama WAJIB berisi tepat satu kata: 'aman' atau 'tidak_aman'.\n"
            "- Jika tidak_aman, baris kedua WAJIB mencantumkan kode kategori yang "
            "dilanggar, dipisahkan koma (contoh: K1, K4).\n"
            "- Jangan tambahkan penjelasan lain. [/INST]"
        )


    def _keyword_check(self, text: str) -> Tuple[bool, Optional[str], str]:
        text_lower = text.lower().strip()
        text_clean = re.sub(r'[\s.\-_*]+', '', text_lower)


        for keyword, cat in self._FLAT_KEYWORDS:
            if keyword in text_clean:
                logger.info(f"Rule-Based BLOCKED: '{keyword}' [cat={cat}] | input: {text[:60]!r}")
                return True, cat, self.EDU_MESSAGES.get(cat, self.DEFAULT_EDU_MSG)


        for pattern, cat in self.TOXIC_PATTERNS_CATEGORIZED:
            if re.search(pattern, text_clean):
                logger.info(f"Rule-Based Pattern BLOCKED: [cat={cat}] | input: {text[:60]!r}")
                return True, cat, self.EDU_MESSAGES.get(cat, self.DEFAULT_EDU_MSG)


        return False, None, ""


    def check(self, text: str) -> Tuple[bool, Optional[str], str]:
        # Layer 1: Rule-Based (SELALU jalan)
        is_toxic, category, edu_msg = self._keyword_check(text)
        if is_toxic:
            return True, category, edu_msg


        # Layer 2: LlamaGuard LLM + Safety Policy (jika tersedia)
        if not self.enabled or self.llm is None:
            logger.warning("LlamaGuard tidak aktif — hanya Rule-Based filter yang digunakan.")
            return False, None, ""


        try:
            prompt = self._build_prompt(text)
            raw    = self.llm.invoke(prompt).strip().lower()
            logger.debug(f"LlamaGuard raw output: {raw!r}")


            lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
            if not lines:
                return False, None, ""


            first_line = lines[0]
            is_unsafe = (
                "tidak_aman" in first_line
                or "tidak aman" in first_line
                or "unsafe" in first_line
            )


            if not is_unsafe:
                return False, None, ""


            category = None
            if len(lines) > 1:
                match = re.search(r'\b(K(?:10|[1-9]))\b', lines[1].upper())
                category = match.group(1) if match else None


            edu_msg = self.EDU_MESSAGES.get(category or "", self.DEFAULT_EDU_MSG)
            logger.info(f"LlamaGuard UNSAFE — kategori: {category} | input: {text[:60]!r}")
            return True, category, edu_msg


        except Exception as e:
            logger.error(f"LlamaGuard error: {e}. Rule-Based sudah dijalankan sebelumnya.")
            return False, None, ""




# ========================================
# DOCUMENT RERANKER
# ========================================


class DocumentReranker:
    def __init__(self):
        try:
            cuda_device = "cuda" if CUDA_AVAILABLE else "cpu"
            self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2',device=cuda_device)
            self.enabled = True
            logger.info("… Cross-encoder reranker initialized")
        except Exception as e:
            logger.warning(f"Reranker gagal: {e}")
            self.model   = None
            self.enabled = False


    def rerank(self, query: str, documents: List[Document], top_n: int = 4) -> List[Document]:
        if not self.enabled or not documents:
            return documents[:top_n]
        try:
            pairs  = [[query, doc.page_content] for doc in documents]
            scores = self.model.predict(pairs)
            ranked = [documents[i] for i in np.argsort(scores)[::-1][:top_n]]
            logger.info(f"Reranked {len(documents)} → top {top_n}")
            return ranked
        except Exception as e:
            logger.error(f"Reranking error: {e}")
            return documents[:top_n]




# ========================================
# RELEVANCE SCORER
# ========================================


class RelevanceScorer:
    def __init__(self, embeddings: HuggingFaceEmbeddings, llm: OllamaLLM):
        self.embeddings = embeddings
        self.llm        = llm


    def cosine_similarity_score(self, text1: str, text2: str) -> float:
        try:
            e1 = np.array(self.embeddings.embed_query(text1)).reshape(1, -1)
            e2 = np.array(self.embeddings.embed_query(text2)).reshape(1, -1)
            return float(cosine_similarity(e1, e2)[0][0])
        except Exception as e:
            logger.error(f"Cosine error: {e}")
            return 0.0


    def semantic_overlap_score(self, question: str, answer: str, context: str) -> float:
        try:
            stopwords = {
                'yang','dan','di','dari','untuk','pada','adalah',
                'ini','itu','dengan','ke','oleh','atau','juga','ada'
            }
            def kw(text):
                words = re.findall(r'\b\w+\b', text.lower())
                return set(w for w in words if w not in stopwords and len(w) > 2)


            q_kw, a_kw, c_kw = kw(question), kw(answer), kw(context)
            score = (len(q_kw & c_kw) / max(len(q_kw), 1) * 0.3 +
                     len(a_kw & c_kw) / max(len(a_kw), 1) * 0.7)
            return round(score, 3)
        except Exception as e:
            logger.error(f"Semantic overlap error: {e}")
            return 0.0


    def llm_judge_score(self, question: str, answer: str, context: str) -> float:
        """
        Metode ke-3: LLM-as-Judge (formula 2.19 proposal, bobot tertinggi w3).
        LLM menilai relevansi jawaban secara kualitatif, menghasilkan skor 0.0-1.0.
        Aktifkan: relevance.methods = ["cosine_similarity","semantic_overlap","llm_judge"]
        """
        try:
            judge_prompt = (
                "Nilai relevansi jawaban berikut terhadap pertanyaan dan konteks.\n"
                "Berikan HANYA angka desimal antara 0.0 dan 1.0 (tanpa penjelasan lain).\n"
                "1.0 = sangat relevan dan sepenuhnya berdasarkan konteks.\n"
                "0.0 = tidak relevan atau tidak berdasarkan konteks.\n\n"
                f"Pertanyaan: {question[:200]}\n"
                f"Konteks: {context[:500]}\n"
                f"Jawaban: {answer[:300]}\n\n"
                "Skor (0.0-1.0):"
            )
            raw   = strip_thinking(self.llm.invoke(judge_prompt).strip())
            match = re.search(r"\b(0(?:\.\d+)?|1(?:\.0*)?)\b", raw)
            return round(min(max(float(match.group()), 0.0), 1.0), 3) if match else 0.0
        except Exception as e:
            logger.error(f"LLM judge error: {e}")
            return 0.0


    def calculate_combined_score(
        self, question: str, answer: str, context: str,
        methods: List[str] = None
    ) -> Dict:
        """
        Hitung relevansi gabungan (formula 2.19 proposal):
          score_final = w1*score_cos + w2*score_sem [+ w3*score_llm]
        """
        if methods is None:
            methods = config.get("relevance.methods", ["cosine_similarity", "semantic_overlap"])


        scores = {}
        if "cosine_similarity" in methods:
            scores["cosine_similarity"] = self.cosine_similarity_score(answer, context)
        if "semantic_overlap" in methods:
            scores["semantic_overlap"]  = self.semantic_overlap_score(question, answer, context)
        if "llm_judge" in methods:
            scores["llm_judge"] = self.llm_judge_score(question, answer, context)


        if "llm_judge" in methods:
            weights = {"cosine_similarity": 0.2, "semantic_overlap": 0.3, "llm_judge": 0.5}
        else:
            weights = {"cosine_similarity": 0.4, "semantic_overlap": 0.6}


        total_w = sum(weights.get(m, 0) for m in methods if m in weights)
        final   = sum(scores.get(m, 0) * weights.get(m, 0) for m in methods if m in weights)
        final   = final / total_w if total_w > 0 else 0


        return {
            "final_score":   round(final, 3),
            "method_scores": scores,
            "weights_used":  {m: weights.get(m) for m in methods if m in weights},
            "is_relevant":   final >= config.get("relevance.threshold", 0.6)
        }




# ========================================
# RAGAS EVALUATOR - Pakai Ollama Lokal
# ========================================


class RAGASEvaluator:


    def __init__(self, hf_embeddings: Optional[HuggingFaceEmbeddings] = None):
        self.available   = RAGAS_AVAILABLE
        self.result_file = config.get("ragas.result_file", "./evaluations/ragas_results.jsonl")
        os.makedirs(os.path.dirname(self.result_file), exist_ok=True)


        if self.available:
            try:
                ollama_llm = OllamaLLM(
                    model=config.get("llm.primary_model", "deepseek-r1:7b"),
                    temperature=0.0,
                    timeout=config.get("llm.timeout", 180)
                )


                self.ragas_llm = LangchainLLMWrapper(ollama_llm)


                if hf_embeddings is not None:
                    self.ragas_emb = LangchainEmbeddingsWrapper(hf_embeddings)
                    logger.info("… RAGASEvaluator: menggunakan HuggingFace embeddings")
                else:
                    # Fallback: gunakan model embedding khusus dari Ollama jika ada
                    ragas_emb_model = config.get("ragas.embedding_model", "nomic-embed-text")
                    try:
                        ollama_emb = OllamaEmbeddings(model=ragas_emb_model)
                        self.ragas_emb = LangchainEmbeddingsWrapper(ollama_emb)
                        logger.info(f"… RAGASEvaluator: menggunakan Ollama embeddings ({ragas_emb_model})")
                    except Exception:
                        # Last resort: wrap OllamaLLM embeddings (tidak ideal)
                        ollama_emb = OllamaEmbeddings(
                            model=config.get("llm.primary_model", "deepseek-r1:7b")
                        )
                        self.ragas_emb = LangchainEmbeddingsWrapper(ollama_emb)
                        logger.warning("⚠️ RAGASEvaluator: menggunakan model generatif sebagai embeddings (tidak ideal)")


                self.metrics_no_gt   = [faithfulness, answer_relevancy]  # context_precision requires reference (ground_truth)
                self.metrics_with_gt = [faithfulness, answer_relevancy, context_precision, context_recall]


                for m in self.metrics_no_gt + [context_recall]:
                    try:
                        # 0.1.x: set directly; 0.2.x: use llm property if exists
                        if hasattr(m, 'llm'):
                            m.llm = self.ragas_llm
                        if hasattr(m, 'embeddings'):
                            m.embeddings = self.ragas_emb
                    except Exception:
                        pass  # Will be configured via evaluate() kwargs


                logger.info("… RAGASEvaluator ready (Ollama lokal — tanpa OpenAI)")
            except Exception as e:
                logger.warning(f"⚠️ RAGAS setup gagal: {e}")
                self.available = False
        else:
            logger.warning("⚠️  RAGASEvaluator tidak tersedia")


    def _build_dataset(self, questions, answers, contexts, ground_truths=None):
        data = {"question": questions, "answer": answers, "contexts": contexts}
        if ground_truths:
            # Kompatibel RAGAS 0.1.x ("ground_truth") dan 0.2+ ("reference")
            data["ground_truth"] = ground_truths
            data["reference"]    = ground_truths
        return Dataset.from_dict(data)


    def _run_evaluate(self, ds, metrics):
        """Wrapper kompatibel RAGAS 0.1.x dan 0.2+."""
        metric_names = [getattr(m, 'name', m.__class__.__name__) for m in metrics]
        logger.info(f"[RAGAS] _run_evaluate: mulai — metrics={metric_names}, n_rows={len(ds)}")
        _t0 = datetime.now()
        try:
            # Coba dengan llm + embeddings (0.1.x dan sebagian 0.2.x)
            result = evaluate(
                ds,
                metrics=metrics,
                llm=self.ragas_llm,
                embeddings=self.ragas_emb
            )
        except TypeError:
            # RAGAS versi tertentu tidak terima llm/embeddings sebagai kwarg evaluate()
            try:
                result = evaluate(ds, metrics=metrics)
            except Exception as e2:
                logger.error(f"[RAGAS] _run_evaluate fallback juga gagal: {e2}")
                raise
        except Exception as e:
            logger.error(f"[RAGAS] _run_evaluate error: {e}")
            raise
        elapsed = (datetime.now() - _t0).total_seconds()
        logger.info(f"[RAGAS] _run_evaluate: selesai dalam {elapsed:.1f}s")
        return result


    def evaluate_single(
        self, question: str, answer: str,
        contexts: List[str], ground_truth: Optional[str] = None
    ) -> Dict:
        if not self.available:
            return {"ragas_available": False}
        logger.info(f"[RAGAS] evaluate_single: start — {question[:60]!r}")
        _t0 = datetime.now()
        try:
            ds      = self._build_dataset([question], [answer], [contexts],
                                          [ground_truth] if ground_truth else None)
            metrics = self.metrics_with_gt if ground_truth else self.metrics_no_gt
            result  = self._run_evaluate(ds, metrics)
            row     = result.to_pandas().iloc[0].to_dict()
            # FIX: Filter out non-numeric columns and handle safely
            exclude_keys = {"question", "answer", "contexts", "ground_truth", "reference"}
            scores  = {}
            for k, v in row.items():
                if k.lower() in exclude_keys:
                    continue
                try:
                    scores[k] = round(float(v), 4) if pd.notna(v) else None
                except (ValueError, TypeError):
                    # Skip non-numeric values
                    pass
            elapsed = (datetime.now() - _t0).total_seconds()
            logger.info(f"[RAGAS] evaluate_single: selesai ({elapsed:.1f}s) — scores={scores}")
            self._save_result(question, answer, contexts, scores, ground_truth)
            return {"ragas_scores": scores, "ragas_available": True}
        except Exception as e:
            logger.error(f"[RAGAS] evaluate_single error: {e}", exc_info=True)
            return {"ragas_available": True, "ragas_error": str(e)}


    def evaluate_batch(
        self, questions, answers, contexts, ground_truths=None
    ) -> Dict:
        if not self.available:
            return {"error": "RAGAS tidak tersedia", "ragas_available": False}
        if not (len(questions) == len(answers) == len(contexts)):
            return {"error": "Panjang questions, answers, contexts harus sama"}
        logger.info(f"[RAGAS] evaluate_batch: start — {len(questions)} question(s), ground_truths={'ada' if ground_truths else 'tidak ada'}")
        _t0 = datetime.now()
        try:
            logger.info(f"[RAGAS] evaluate_batch: membangun dataset...")
            ds      = self._build_dataset(questions, answers, contexts, ground_truths)
            metrics = self.metrics_with_gt if ground_truths else self.metrics_no_gt
            logger.info(f"[RAGAS] evaluate_batch: menjalankan evaluasi RAGAS (ini bisa 2-10 menit)...")
            result  = self._run_evaluate(ds, metrics)
            df      = result.to_pandas()


            # Filter hanya numeric columns (metrics), exclude text columns
            metric_cols = [c for c in df.columns
                           if c.lower() not in ("question", "answer", "contexts",
                                        "ground_truth", "reference")
                           and df[c].dtype in [float, int, "float64", "int64"]]


            per_sample = []
            for i, row in df.iterrows():
                # FIX: Only convert numeric columns to float
                s = {}
                for k in metric_cols:
                    try:
                        val = row[k]
                        s[k] = round(float(val), 4) if pd.notna(val) else None
                    except (ValueError, TypeError):
                        s[k] = None
                s["question"] = questions[i]
                per_sample.append(s)


            numeric_cols = [c for c in metric_cols if df[c].dtype in [float, int, "float64", "int64"]]
            aggregate    = {col: round(float(df[col].mean()), 4) for col in numeric_cols if col in df.columns}


            elapsed = (datetime.now() - _t0).total_seconds()
            logger.info(f"[RAGAS] evaluate_batch: selesai ({elapsed:.1f}s) — aggregate={aggregate}")
            return {
                "ragas_available":   True,
                "total_evaluated":   len(questions),
                "aggregate_scores":  aggregate,
                "per_sample_scores": per_sample,
                "timestamp":         datetime.now().isoformat()
            }
        except Exception as e:
            logger.error(f"[RAGAS] evaluate_batch error: {e}", exc_info=True)
            return {"error": str(e), "ragas_available": True}


    def _save_result(self, question, answer, contexts, scores, ground_truth):
        try:
            record = {
                "timestamp":        datetime.now().isoformat(),
                "question":         question,
                "answer":           answer[:300],
                "contexts_preview": [c[:150] for c in contexts],
                "ground_truth":     ground_truth,
                "scores":           scores
            }
            with open(self.result_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        except Exception as e:
            logger.error(f"Gagal simpan RAGAS: {e}")


    def get_summary(self) -> Dict:
        if not os.path.exists(self.result_file):
            return {"message": "Belum ada evaluasi RAGAS"}
        all_scores: Dict[str, List[float]] = defaultdict(list)
        total = 0
        try:
            with open(self.result_file, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        rec = json.loads(line)
                        for k, v in rec.get("scores", {}).items():
                            # Gunakan pure-Python NaN check (v != v hanya True untuk float NaN)
                            # sehingga tidak bergantung pada pandas yang mungkin tidak terinstall
                            if isinstance(v, (int, float)) and v == v:
                                all_scores[k].append(float(v))
                        total += 1
                    except Exception:
                        continue
            return {
                "total_ragas_evaluations": total,
                "average_scores": {k: round(np.mean(v), 4) for k, v in all_scores.items() if v},
                "result_file":    self.result_file
            }
        except Exception as e:
            return {"error": str(e)}




# ========================================
# ANTI-HALLUCINATION GUARD
# ========================================


class AntiHallucinationGuard:
    NO_INFO_PATTERNS = [
        r"tidak ada dalam buku",
        r"tidak tersedia dalam materi",
        r"informasi ini tidak ada",
        r"maaf.*tidak.*buku",
        r"tidak ditemukan dalam",
        r"buku.*tidak.*membahas",
        r"tidak dibahas dalam materi",
    ]
    HALLUCINATION_PATTERNS = [
        r"menurut para ahli",
        r"secara umum diketahui",
        r"dari berbagai sumber",
        r"berdasarkan penelitian",
        r"wikipedia",
        r"umumnya",
    ]


    @classmethod
    def is_valid_no_info(cls, answer: str) -> bool:
        return any(re.search(p, answer.lower()) for p in cls.NO_INFO_PATTERNS)


    @classmethod
    def has_hallucination_signal(cls, answer: str) -> bool:
        return any(re.search(p, answer.lower()) for p in cls.HALLUCINATION_PATTERNS)


    @classmethod
    def compute_token_overlap(cls, answer: str, context: str) -> float:
        stopwords = {
            'yang','dan','di','dari','untuk','pada','adalah','ini','itu',
            'dengan','ke','oleh','atau','juga','saya','kami','kita','ada',
            'tidak','akan','sudah','bisa','lebih','sangat','dalam','jika',
            'karena','maka','a','an','the','is','are','was','were','be','been'
        }


        def tok(text):
            words = re.findall(r'\b\w+\b', text.lower())
            return [w for w in words if w not in stopwords and len(w) > 2]


        def bigrams(tokens):
            return set(zip(tokens, tokens[1:])) if len(tokens) > 1 else set()


        a_tok = tok(answer)
        c_tok = tok(context)
        uni   = len(set(a_tok) & set(c_tok)) / max(len(set(a_tok)), 1)
        a_bi, c_bi = bigrams(a_tok), bigrams(c_tok)
        bi    = len(a_bi & c_bi) / max(len(a_bi), 1) if a_bi else 0.0
        return round(0.5 * uni + 0.5 * bi, 4)


    @classmethod
    def check_and_sanitize(cls, answer: str, context: str, question: str) -> Tuple[str, Dict]:
        if cls.is_valid_no_info(answer):
            return answer, {"passed": True, "reason": "valid_no_info", "overlap": 0.0, "action": "accepted"}


        overlap    = cls.compute_token_overlap(answer, context)
        has_hallu  = cls.has_hallucination_signal(answer)
        strict     = config.get("anti_hallucination.strict_mode", True)
        threshold  = config.get("anti_hallucination.overlap_refuse_threshold", 0.30)


        # FIX: Cek juga overlap pertanyaan↔konteks.
        # Jika pertanyaan COCOK dengan konteks (materi ada di buku),
        # jangan langsung tolak — mungkin LLM hanya parafrase/jawab singkat.
        question_context_overlap = cls.compute_token_overlap(question, context)


        # STRICT: Jika overlap sangat rendah, TOLAK jawaban (kemungkinan besar di luar materi)
        # TAPI: jika pertanyaan sangat cocok dgn konteks (>0.3), berarti materi ADA di buku,
        # hanya jawaban LLM yang kurang verbatim — jangan blokir.
        if overlap < threshold and strict:
            # Bypass: jika question↔context overlap tinggi, materi ada di buku
            if question_context_overlap >= 0.25:
                logger.info(
                    f"Guard BYPASS: answer overlap rendah ({overlap:.2f}) tapi "
                    f"question↔context tinggi ({question_context_overlap:.2f}) — materi ada di buku"
                )
                disclaimer = "\n\n⚠️ *Catatan: Jawaban mungkin kurang sesuai dengan materi buku.*"
                return answer + disclaimer, {
                    "passed": False, "reason": "low_overlap_but_relevant_context",
                    "overlap": overlap, "question_context_overlap": question_context_overlap,
                    "action": "warned"
                }


            safe = ("Maaf, pertanyaan kamu sepertinya di luar materi buku yang diunggah. "
                    "Coba tanyakan hal-hal yang berkaitan dengan isi buku pelajaranmu ya! 📖")
            return safe, {"passed": False, "reason": "out_of_context", "overlap": overlap, "action": "replaced"}


        # WARNING: Overlap di antara threshold dan min_context_overlap
        min_overlap = config.get("anti_hallucination.min_context_overlap", 0.30)
        if overlap < min_overlap:
            disclaimer = "\n\n⚠️ *Catatan: Jawaban mungkin kurang sesuai dengan materi buku.*"
            return answer + disclaimer, {"passed": False, "reason": "low_overlap", "overlap": overlap, "action": "warned"}


        return answer, {"passed": True, "reason": "ok", "overlap": overlap, "action": "accepted"}




# ========================================
# QUIZ VALIDATOR
# ========================================


class QuizValidator:
    LETTERS = ['A', 'B', 'C', 'D']


    @classmethod
    def _compute_overlap(cls, text: str, context: str) -> float:
        """Hitung token overlap antara teks soal/penjelasan dengan konteks sumber."""
        stopwords = {
            'yang','dan','di','dari','untuk','pada','adalah','ini','itu',
            'dengan','ke','oleh','atau','juga','ada','tidak','akan','sudah',
            'bisa','lebih','sangat','dalam','jika','karena','maka','the',
            'a','an','is','are','of','to','in','for','soal','jawaban',
            'berikut','pilihan','benar','salah'
        }
        def tok(t):
            words = re.findall(r'\b\w+\b', t.lower())
            return set(w for w in words if w not in stopwords and len(w) > 2)
        t_tok = tok(text)
        c_tok = tok(context)
        if not t_tok:
            return 0.0
        return len(t_tok & c_tok) / len(t_tok)


    @classmethod
    def validate_and_fix(cls, q: Dict, context: str = "") -> Optional[Dict]:
        """Validasi dan perbaiki satu soal quiz. Tolak jika tidak relevan dengan konteks."""
        if not isinstance(q, dict):
            return None


        required = ["soal", "opsi", "kunci", "penjelasan"]
        if not all(f in q for f in required):
            return None


        if not isinstance(q["opsi"], list) or len(q["opsi"]) != 4:
            return None


        kunci = str(q["kunci"]).strip().upper()
        if kunci not in cls.LETTERS:
            return None


        opsi_lower = [str(o).lower().strip() for o in q["opsi"]]
        if len(set(opsi_lower)) < 3:
            return None


        if not q["soal"].strip() or not q["penjelasan"].strip():
            return None


        soal = q["soal"].strip()
        if not soal.endswith('?'):
            kata_tanya = ['apa','siapa','bagaimana','mengapa','kapan','di mana',
                          'berapa','sebutkan','jelaskan','manakah']
            if any(kw in soal.lower() for kw in kata_tanya):
                soal = soal + '?'


        # Semantic validation: tolak soal yang isinya tidak nyambung dengan konteks
        if context:
            combined_text = soal + " " + q["penjelasan"]
            overlap = cls._compute_overlap(combined_text, context)
            if overlap < 0.15:
                logger.warning(
                    f"Quiz soal DITOLAK (overlap={overlap:.2f} < 0.15): {soal[:60]}..."
                )
                return None


        return {
            "soal":       soal,
            "opsi":       [str(o).strip() for o in q["opsi"]],
            "kunci":      kunci,
            "penjelasan": q["penjelasan"].strip()
        }




# ========================================
# MAIN ASSISTANT
# ========================================






class UltimateAssistantLCEL:


    def __init__(self):
        device_for_embeddings = "cuda" if CUDA_AVAILABLE else "cpu"
        self.embeddings = HuggingFaceEmbeddings(
            model_name=config.get("embedding.model_name",
                                  "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"),
            model_kwargs={'device': device_for_embeddings},
            encode_kwargs={'normalize_embeddings': True}
        )


        self.llm_qa = OllamaLLM(
            model=config.get("llm.primary_model", "deepseek-r1:7b"),
            temperature=config.get("llm.temperature.qa", 0.1),
            timeout=config.get("llm.timeout", 180),
            base_url=config.get("llm.ollama_base_url", "http://localhost:11434")
        )
        self.llm_quiz = OllamaLLM(
            model=config.get("llm.primary_model", "deepseek-r1:7b"),
            temperature=config.get("llm.temperature.quiz", 0.3),
            timeout=config.get("llm.timeout", 180),
            base_url=config.get("llm.ollama_base_url", "http://localhost:11434")
        )


        logger.info(f"LLMs initialized: {config.get('llm.primary_model')}")


        self.text_splitter      = SemanticChunker(self.embeddings)
        self.reranker           = DocumentReranker()
        self.relevance_scorer   = RelevanceScorer(self.embeddings, self.llm_qa)
        # FIX: Teruskan HF embeddings ke RAGASEvaluator agar tidak pakai model generatif
        self.ragas_evaluator    = RAGASEvaluator(hf_embeddings=self.embeddings)
        self.anti_hallucination = AntiHallucinationGuard()
        self.quiz_validator     = QuizValidator()
        self.toxicity_filter    = LlamaGuardFilter()


        # FIX: Gunakan threading.Lock untuk keamanan akses stats konkuren
        self._stats_lock = threading.Lock()
        self.stats       = defaultdict(int)


        self._last_quiz_context = ""


        self.db_dir   = config.get("paths.vector_db",   "./vector_db_lcel")
        self.eval_dir = config.get("paths.evaluations", "./evaluations")


        self.vector_db  = None
        self.retriever  = None
        self.rag_chain  = None
        self.quiz_chain = None


    def _increment_stat(self, key: str, amount: int = 1) -> None:
        """Thread-safe increment untuk stats counter."""
        with self._stats_lock:
            self.stats[key] += amount


    # ─── RETRIEVER ───────────────────────────────────────────────


    def _build_retriever(self):
        if self.vector_db is None:
            if not os.path.exists(self.db_dir):
                return None
            self.vector_db = Chroma(
                persist_directory=self.db_dir,
                embedding_function=self.embeddings
            )
        self.retriever = self.vector_db.as_retriever(
            search_type=config.get("retrieval.method", "mmr"),
            search_kwargs={
                "k":           config.get("retrieval.top_k",       8),
                "fetch_k":     config.get("retrieval.fetch_k",     30),
                "lambda_mult": config.get("retrieval.lambda_mult", 0.7)
            }
        )
        return self.retriever


    # ─── RAG CHAIN ───────────────────────────────────────────────


    def _build_rag_chain(self):
        if self.retriever is None:
            self._build_retriever()
        if self.retriever is None:
            return None


        qa_template = """SISTEM: Kamu adalah Guru SD Indonesia yang ramah, jujur, dan sangat teliti.
Kamu AHLI membaca dan memahami isi buku pelajaran. WAJIB menggunakan Bahasa Indonesia.


════════════════════════════════════════
ATURAN KERAS (WAJIB DIIKUTI):
════════════════════════════════════════
1. Jawab HANYA dari MATERI di bawah ini.
2. DILARANG menggunakan pengetahuan umum, internet, atau sumber lain.
3. Jika informasi TIDAK ADA di materi → jawab:
   "Maaf, informasi ini tidak ada dalam buku yang saya punya."
4. JANGAN mengarang atau menduga-duga.
5. Bahasa: sederhana, mudah dipahami anak SD.
6. Panjang: maksimal 3-4 kalimat, singkat dan padat.
7. Mulai dengan "Berdasarkan materi..." jika informasi tersedia.


════════════════════════════════════════
MATERI DARI BUKU (SATU-SATUNYA SUMBER):
════════════════════════════════════════
{context}


════════════════════════════════════════
PERTANYAAN SISWA:
{question}
════════════════════════════════════════


JAWABAN GURU (Bahasa Indonesia, jelaskan dengan lengkap dan mudah dipahami):"""


        qa_prompt = ChatPromptTemplate.from_template(qa_template)


        def format_docs(docs: List[Document]) -> str:
            return "\n\n---\n\n".join(
                f"[Bagian {i+1}]\n{doc.page_content}"
                for i, doc in enumerate(docs)
            )


        def rerank_step(inputs: Dict) -> Dict:
            if config.get("retrieval.rerank.enabled", True):
                top_n    = config.get("retrieval.rerank.top_n", 4)
                reranked = self.reranker.rerank(inputs["question"], inputs["docs"], top_n)
                return {"docs": reranked, "question": inputs["question"]}
            return inputs


        self.rag_chain = (
            RunnableParallel({
                "docs":     self.retriever,
                "question": RunnablePassthrough()
            })
            | RunnableLambda(rerank_step)
            | RunnableParallel({
                "context":  lambda x: format_docs(x["docs"]),
                "question": lambda x: x["question"],
                "docs":     lambda x: x["docs"]
            })
            | {
                "answer":   qa_prompt | self.llm_qa | StrOutputParser() | RunnableLambda(strip_thinking),
                "context":  lambda x: x["context"],
                "question": lambda x: x["question"],
                "docs":     lambda x: x["docs"]
            }
        )


        logger.info("… RAG Chain built")
        return self.rag_chain


    # ─── QUIZ CHAIN ──────────────────────────────────────────────


    def _build_quiz_chain(self):
        if self.vector_db is None:
            self._build_retriever()
        if self.vector_db is None:
            return None


        quiz_template = """SISTEM: Kamu adalah Guru SD Indonesia yang AHLI membuat soal quiz berkualitas.
WAJIB menggunakan Bahasa Indonesia. DILARANG menggunakan bahasa Inggris.


TUGAS: Baca SELURUH materi di bawah dengan teliti, lalu buat TEPAT {num_questions} soal pilihan ganda.
Soal harus menguji PEMAHAMAN siswa, bukan hanya hafalan kata.


════════════════════════════════════════
MATERI DARI BUKU:
════════════════════════════════════════
{context}
════════════════════════════════════════


ATURAN WAJIB:
1. Soal HARUS bisa dijawab langsung dari materi di atas.
2. DILARANG membuat soal dari pengetahuan umum di luar materi.
3. Gunakan Bahasa Indonesia yang sederhana untuk anak SD.
4. opsi[0]=A, opsi[1]=B, opsi[2]=C, opsi[3]=D.
5. "kunci" adalah huruf jawaban BENAR: A/B/C/D.
6. PERIKSA ULANG: kunci="A" → opsi[0] benar, kunci="B" → opsi[1] benar, dst.
7. "penjelasan" harus mengutip kalimat dari materi di atas.


FORMAT JSON (HANYA JSON, TANPA TEKS LAIN):
{{
  "questions": [
    {{
      "soal": "Pertanyaan dalam Bahasa Indonesia?",
      "opsi": ["Jawaban benar (A)", "Pengecoh 1 (B)", "Pengecoh 2 (C)", "Pengecoh 3 (D)"],
      "kunci": "A",
      "penjelasan": "Berdasarkan materi: [kutip kalimat dari materi]."
    }}
  ]
}}


HANYA JSON:"""


        quiz_prompt = ChatPromptTemplate.from_template(quiz_template)


        def sample_context_for_quiz(inputs: Dict) -> Dict:
            """
            Ambil sampel acak chunk dari ChromaDB.
            Mendukung filter bab (metadata) sesuai proposal Bab 3.
            """
            num_questions = inputs.get("num_questions", 3)
            bab_filter    = inputs.get("bab", None)
            chunks_per_q  = config.get("quiz.chunks_per_question", 3)
            chunks_needed = num_questions * chunks_per_q


            all_docs  = self.vector_db.get()  # Get ALL docs tanpa limit
            if not all_docs or "documents" not in all_docs:
                raise ValueError("Tidak ada dokumen dalam database.")


            raw_docs  = all_docs["documents"]
            metadatas = all_docs.get("metadatas", [{}] * len(raw_docs))


            if bab_filter is not None:
                if isinstance(bab_filter, list):
                    bab_lower = [str(b).lower() for b in bab_filter]
                    pairs = [(d, m) for d, m in zip(raw_docs, metadatas)
                             if str(m.get("bab", "")).lower() in bab_lower]
                else:
                    target = str(bab_filter).lower()
                    pairs  = [(d, m) for d, m in zip(raw_docs, metadatas)
                              if str(m.get("bab", "")).lower() == target]
                filtered = [d for d, _ in pairs]
                if filtered:
                    raw_docs = filtered
                    logger.info(f"Filter bab='{bab_filter}': {len(raw_docs)} chunk tersedia")
                else:
                    logger.warning(
                        f"⚠️ Filter bab='{bab_filter}' tidak menemukan chunk dengan metadata bab. "
                        f"Fallback ke semua {len(raw_docs)} chunk."
                    )


            # Filter: hanya ambil chunks bermakna (min 50 chars untuk kualitas konteks)
            docs = [d for d in raw_docs if len(d.strip()) > 50]
            if not docs:
                raise ValueError("Tidak ada chunk dengan panjang memadai (min 50 karakter).")


            if len(docs) < chunks_needed:
                # Jangan error, kurangi chunks_needed ke jumlah yang tersedia
                logger.warning(
                    f"Chunk tersedia ({len(docs)}) kurang dari ideal "
                    f"({chunks_needed} = {num_questions} soal x {chunks_per_q} chunk/soal). "
                    f"Menggunakan semua {len(docs)} chunk yang tersedia."
                )
                chunks_needed = len(docs)


            if len(docs) <= chunks_needed:
                sampled = docs
            else:
                # Random sample (bukan sequential) untuk variasi soal lebih baik
                sampled = random.sample(docs, chunks_needed)


            context = ("\n\n---\n\n".join(sampled))[:12000]


            # Simpan untuk dipakai QuizValidator
            self._last_quiz_context = context


            logger.info(f"Quiz context: {len(sampled)} chunks, {len(context)} chars")
            return {"context": context, "num_questions": num_questions}


        def parse_quiz_output(text: str) -> Dict:
            text = clean_json_output(text)
            try:
                return json.loads(text)
            except Exception:
                pass
            try:
                match = re.search(r'\{.*\}', text, re.DOTALL)
                if match:
                    return json.loads(match.group())
            except Exception:
                pass
            try:
                match = re.search(r'"questions"\s*:\s*\[.*?\]', text, re.DOTALL)
                if match:
                    return json.loads('{' + match.group() + '}')
            except Exception:
                pass
            logger.error(f"Semua parsing gagal. Output: {text[:300]}")
            return {"questions": []}


        self.quiz_chain = (
            RunnableLambda(sample_context_for_quiz)
            | quiz_prompt
            | self.llm_quiz
            | StrOutputParser()
            | RunnableLambda(parse_quiz_output)
        )


        logger.info("… Quiz Chain built")
        return self.quiz_chain


    # ─── OCR IMAGE PREPROCESSING ────────────────────────────────


    @staticmethod
    def _preprocess_image_for_ocr(pil_img):
        """
        Preprocessing gambar dengan OpenCV sebelum OCR.
        Pipeline: RGB → Grayscale → Denoise → Adaptive Threshold → Binarisasi.
        Meningkatkan akurasi Tesseract secara signifikan untuk PDF scan.
        Mengembalikan list variasi gambar (preprocessed) untuk dicoba OCR.
        """
        import numpy as np


        if not CV2_AVAILABLE:
            return [pil_img]  # fallback tanpa preprocessing


        # PIL Image → numpy array (OpenCV format BGR)
        img_np = np.array(pil_img)
        if len(img_np.shape) == 3:
            gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        else:
            gray = img_np


        results = []


        # Strategi 1: Denoise + Adaptive Threshold (terbaik untuk scan)
        try:
            denoised = cv2.fastNlMeansDenoising(gray, None, h=10, templateWindowSize=7, searchWindowSize=21)
            adaptive = cv2.adaptiveThreshold(
                denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, blockSize=31, C=10
            )
            results.append(adaptive)
        except Exception:
            pass


        # Strategi 2: OTSU Threshold (bagus untuk kontras rendah)
        try:
            blur = cv2.GaussianBlur(gray, (5, 5), 0)
            _, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            results.append(otsu)
        except Exception:
            pass


        # Strategi 3: Grayscale + contrast enhancement (CLAHE)
        try:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            enhanced = clahe.apply(gray)
            results.append(enhanced)
        except Exception:
            pass


        if not results:
            return [pil_img]  # fallback ke gambar asli jika semua strategi gagal


        # Konversi numpy arrays kembali ke PIL Images
        from PIL import Image
        pil_results = []
        for arr in results:
            pil_results.append(Image.fromarray(arr))


        return pil_results


    # ─── OCR FALLBACK (untuk PDF berbasis gambar/scan) ───────────


    def _ocr_fallback(self, file_path: str) -> List[Document]:
        """
        Fallback OCR: konversi PDF → gambar → preprocessing → pytesseract.
        Pipeline: PDF → pdf2image(300 DPI) → OpenCV preprocessing → Tesseract OCR.
        Mencoba beberapa strategi preprocessing dan PSM mode, ambil hasil terbaik.
        """
        if not PYTESSERACT_AVAILABLE or not config.get("pdf.tesseract_available", False):
            logger.warning("Tesseract/PyTesseract tidak tersedia — OCR fallback dilewati")
            return []


        if not PDF2IMAGE_AVAILABLE:
            logger.warning("pdf2image tidak tersedia — OCR fallback dilewati")
            return []


        ocr_lang = config.get("pdf.ocr_languages", "ind+eng")
        logger.info(f"🔄 OCR Fallback dimulai — lang={ocr_lang} | file={file_path}")


        # PSM modes yang dicoba (pick best result):
        #   6 = Assume a single uniform block of text
        #   3 = Fully automatic page segmentation (default)
        psm_modes = [6, 3]


        try:
            # Konversi PDF → gambar per halaman (300 DPI untuk kualitas OCR)
            images = convert_from_path(file_path, dpi=300)
            logger.info(f"   PDF → {len(images)} halaman gambar")


            docs = []
            for i, img in enumerate(images):
                best_text = ""


                # Coba preprocessing variants
                preprocessed_imgs = self._preprocess_image_for_ocr(img)


                for p_img in preprocessed_imgs:
                    for psm in psm_modes:
                        try:
                            custom_config = f'--oem 3 --psm {psm}'
                            text = pytesseract.image_to_string(
                                p_img, lang=ocr_lang, config=custom_config
                            )
                            text = text.strip()
                            # Ambil hasil dengan teks terpanjang (heuristik kualitas)
                            if len(text) > len(best_text):
                                best_text = text
                        except Exception:
                            continue


                # Juga coba gambar asli tanpa preprocessing (sebagai baseline)
                try:
                    raw_text = pytesseract.image_to_string(img, lang=ocr_lang)
                    if len(raw_text.strip()) > len(best_text):
                        best_text = raw_text.strip()
                except Exception:
                    pass


                # Post-processing: bersihkan karakter noise dari OCR
                best_text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', best_text)
                best_text = re.sub(r'\n{3,}', '\n\n', best_text)  # kurangi newline berlebihan
                best_text = re.sub(r'[ ]{3,}', ' ', best_text)    # kurangi spasi berlebihan


                if best_text and len(best_text) > 20:  # minimal 20 karakter bermakna
                    docs.append(Document(
                        page_content=best_text,
                        metadata={
                            "source": os.path.basename(file_path),
                            "page_number": i + 1,
                            "category": "NarrativeText",
                            "ocr_fallback": True,
                            "ocr_preprocessed": CV2_AVAILABLE
                        }
                    ))
                    logger.info(f"   Halaman {i+1}: {len(best_text)} chars extracted …")
                else:
                    logger.warning(f"   Halaman {i+1}: terlalu sedikit teks ({len(best_text)} chars), skip")


            logger.info(f"… OCR Fallback selesai: {len(docs)}/{len(images)} halaman dengan konten")
            return docs


        except Exception as e:
            logger.error(f"❌ OCR Fallback gagal: {e}", exc_info=True)
            return []


    # ─── INGEST (HANYA GURU) ─────────────────────────────────────


    def _simple_text_extraction(self, file_path: str) -> list:
        """
        Simple text extraction fallback: coba pypdf, pdfplumber, kemudian PyPDF2.
        Fallback terakhir ketika Unstructured dan OCR gagal.
        """
        from langchain_core.documents import Document as _Doc
        docs = []
        methods_tried = []


        # Coba 1: pypdf (modern, sudah ada di requirements)
        try:
            from pypdf import PdfReader as _PdfReader
            logger.info(f"[Attempt 1/3] Simple text extraction (pypdf): {file_path}")
            reader = _PdfReader(file_path)
            for page_num, page in enumerate(reader.pages):
                try:
                    text = (page.extract_text() or "").strip()
                    if text and len(text) > 20:
                        docs.append(_Doc(
                            page_content=text,
                            metadata={"source": os.path.basename(file_path), "page_number": page_num + 1, "category": "NarrativeText"}
                        ))
                except Exception:
                    continue
            if docs:
                logger.info(f"✓ pypdf OK: {len(docs)} pages, {sum(len(d.page_content) for d in docs)} chars")
                return docs
            methods_tried.append("pypdf (0 docs)")
        except ImportError as e:
            logger.warning(f"✗ pypdf not installed: {e}")
            methods_tried.append(f"pypdf (not installed)")
        except Exception as e:
            logger.warning(f"✗ pypdf error: {e}")
            methods_tried.append(f"pypdf ({type(e).__name__})")


        # Coba 2: pdfplumber
        try:
            import pdfplumber
            logger.info(f"[Attempt 2/3] Simple text extraction (pdfplumber): {file_path}")
            with pdfplumber.open(file_path) as pdf:
                for page_num, page in enumerate(pdf.pages):
                    try:
                        text = page.extract_text() or ""
                        tables = page.extract_tables() or []
                        table_text = ""
                        for table in tables:
                            for row in table:
                                table_text += " | ".join(str(cell) if cell else "" for cell in row) + "\n"
                        full_text = (text + "\n" + table_text).strip()
                        if full_text and len(full_text) > 20:
                            docs.append(_Doc(
                                page_content=full_text,
                                metadata={"source": os.path.basename(file_path), "page_number": page_num + 1, "category": "NarrativeText"}
                            ))
                    except Exception:
                        continue
            if docs:
                logger.info(f"✓ pdfplumber OK: {len(docs)} pages, {sum(len(d.page_content) for d in docs)} chars")
                return docs
            methods_tried.append("pdfplumber (0 docs)")
        except ImportError as e:
            logger.warning(f"✗ pdfplumber not installed: {e}")
            methods_tried.append(f"pdfplumber (not installed)")
        except Exception as e:
            logger.warning(f"✗ pdfplumber error: {e}")
            methods_tried.append(f"pdfplumber ({type(e).__name__})")


        # Coba 3: PyPDF2 (legacy fallback)
        try:
            from PyPDF2 import PdfReader
            logger.info(f"[Attempt 3/3] Simple text extraction (PyPDF2): {file_path}")
            reader = PdfReader(file_path)
            for page_num, page in enumerate(reader.pages):
                try:
                    text = (page.extract_text() or "").strip()
                    if text and len(text) > 20:
                        docs.append(_Doc(
                            page_content=text,
                            metadata={"source": os.path.basename(file_path), "page_number": page_num + 1, "category": "NarrativeText"}
                        ))
                except Exception:
                    continue
            if docs:
                logger.info(f"✓ PyPDF2 OK: {len(docs)} pages, {sum(len(d.page_content) for d in docs)} chars")
                return docs
            methods_tried.append("PyPDF2 (0 docs)")
        except ImportError as e:
            logger.warning(f"✗ PyPDF2 not installed: {e}")
            methods_tried.append(f"PyPDF2 (not installed)")
        except Exception as e:
            logger.warning(f"✗ PyPDF2 error: {e}")
            methods_tried.append(f"PyPDF2 ({type(e).__name__})")


        # All methods failed
        logger.error(f"⚠️  All simple text extraction methods failed: {' → '.join(methods_tried)}")
        logger.error(f"   Install missing packages: pip install pdfplumber PyPDF2")
        logger.error(f"   Or install Tesseract-OCR for scan/image PDFs")
        return []


    @retry(
        stop=stop_after_attempt(2),
        wait=wait_exponential(min=2, max=8),
        retry=retry_if_not_exception_type(HTTPException)
    )

    def _ingest_isolated(self, file_path: str, target_db_dir: str) -> Dict:
        """
        Ingest a PDF into a SEPARATE directory WITHOUT modifying the current
        assistant state (vector_db, retriever, rag_chain, quiz_chain).

        This allows concurrent chat/quiz to keep working on the old book
        while the new book is being processed.

        Returns dict with result info + '_new_vector_db' key.
        """
        logger.info(f"[ISOLATED INGEST] Start: {file_path} -> {target_db_dir}")
        start_time = datetime.now()

        if not os.path.exists(file_path):
            raise ValueError(f"File tidak ditemukan: {file_path}")
        if not file_path.lower().endswith('.pdf'):
            raise ValueError("File harus PDF")

        # Clean target dir if it has leftover data
        if os.path.exists(target_db_dir):
            _force_rmtree(target_db_dir)
        os.makedirs(target_db_dir, exist_ok=True)

        # ── STEP 1: PDF extraction ───────────────────────────────────
        step_start = datetime.now()
        tesseract_ready = config.get("pdf.tesseract_available", False)
        base_languages = config.get("pdf.languages", ["ind", "eng"])
        strategy = config.get("pdf.strategy", "hi_res")

        loader_kwargs = {
            "file_path": file_path,
            "strategy": strategy,
            "mode": "elements",
            "extract_images_in_pdf": False,
            "infer_table_structure": False,
            "languages": base_languages,
        }

        loader = UnstructuredPDFLoader(**loader_kwargs)
        try:
            docs = loader.load()
        except Exception as load_err:
            if "tesseract" in str(load_err).lower():
                logger.warning("[ISOLATED] Unstructured gagal (tesseract). Retry tanpa OCR.")
                loader = UnstructuredPDFLoader(
                    file_path, strategy="fast", mode="elements",
                    extract_images_in_pdf=False, infer_table_structure=False,
                    languages=base_languages,
                )
                docs = loader.load()
            else:
                raise

        fast_elapsed = (datetime.now() - step_start).total_seconds()
        logger.info(f"[ISOLATED PERF] Extraction: {fast_elapsed:.1f}s -> {len(docs)} elemen")

        if not docs:
            logger.warning("[ISOLATED] 0 elements, trying simple text extraction...")
            docs = self._simple_text_extraction(file_path)

        # ── STEP 2: Quality check + OCR fallback ─────────────────────
        total_chars = sum(len(d.page_content.strip()) for d in docs) if docs else 0
        file_size_kb = os.path.getsize(file_path) / 1024
        min_chars = config.get("pdf.ocr_min_chars_per_page", 50)
        use_ocr_fallback = config.get("pdf.ocr_fallback", True) and tesseract_ready

        is_likely_scanned = (
            (total_chars < min_chars * 5 and file_size_kb > 100)
            or total_chars < 100
        )

        if is_likely_scanned and use_ocr_fallback:
            logger.warning(f"[ISOLATED] PDF kemungkinan scan: {total_chars} chars. OCR fallback...")
            ocr_docs = self._ocr_fallback(file_path)
            if ocr_docs and sum(len(d.page_content) for d in ocr_docs) > total_chars:
                docs = ocr_docs
        elif not docs or total_chars == 0:
            if use_ocr_fallback:
                docs = self._ocr_fallback(file_path)
                if docs:
                    total_chars = sum(len(d.page_content.strip()) for d in docs)
            if not docs:
                docs = self._simple_text_extraction(file_path)
                if docs:
                    total_chars = sum(len(d.page_content.strip()) for d in docs)
            if not docs or total_chars == 0:
                raise ValueError(f"Gagal extract konten dari PDF: {os.path.basename(file_path)}")

        # ── STEP 3: Chunking ─────────────────────────────────────────
        chunks = self.text_splitter.split_documents(docs)
        logger.info(f"[ISOLATED] {len(chunks)} semantic chunks")
        chunks = filter_complex_metadata(chunks)

        # Tag bab metadata
        BAB_PATTERN = re.compile(
            r'(?i)^(?:bab|chapter|bagian|unit|tema|pelajaran|topik)\s+(\w+)'
            r'|^(BAB|CHAPTER|UNIT|TEMA|PELAJARAN)\s+(\w+)'
        )
        current_bab = "0"
        for chunk in chunks:
            text = chunk.page_content.strip()
            cat = chunk.metadata.get("category", "")
            matched = BAB_PATTERN.match(text)
            is_heading = cat in ("Title", "Header") or (cat in ("UncategorizedText", "NarrativeText") and len(text) < 80)
            if is_heading and matched:
                bab_val = next(
                    (g for g in matched.groups() if g and g.lower() not in ("bab", "chapter", "bagian")),
                    None
                )
                if bab_val:
                    current_bab = bab_val.lower()
            chunk.metadata["bab"] = current_bab

        # ── STEP 4: Create ChromaDB in target dir ─────────────────────
        try:
            new_vector_db = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                persist_directory=target_db_dir
            )
        except Exception as _chroma_err:
            _err_str = str(_chroma_err)
            if "bindings" in _err_str or "RustBindings" in _err_str or "sqlite" in _err_str.lower():
                logger.warning(f"[ISOLATED] ChromaDB error ({_err_str[:80]}). Wiping and retrying...")
                _sqlite = os.path.join(target_db_dir, "chroma.sqlite3")
                if os.path.exists(_sqlite):
                    os.remove(_sqlite)
                for _entry in os.scandir(target_db_dir):
                    if _entry.is_dir():
                        _force_rmtree(_entry.path)
                new_vector_db = Chroma.from_documents(
                    documents=chunks,
                    embedding=self.embeddings,
                    persist_directory=target_db_dir
                )
            else:
                raise

        avg_size = sum(len(c.page_content) for c in chunks) / len(chunks) if chunks else 0
        proc_time = (datetime.now() - start_time).total_seconds()

        result = {
            "status":          "success",
            "total_elements":  len(docs),
            "total_chunks":    len(chunks),
            "avg_chunk_size":  round(avg_size, 2),
            "processing_time": round(proc_time, 2),
            "_new_vector_db":  new_vector_db,
            "timestamp":       datetime.now().isoformat()
        }
        logger.info(f"[ISOLATED INGEST] Done: {len(chunks)} chunks in {proc_time:.1f}s")
        self._increment_stat("total_ingestions")
        return result

    def ingest(self, file_path: str) -> Dict:
        logger.info(f"Mulai ingestion: {file_path}")
        start_time = datetime.now()


        if not os.path.exists(file_path):
            raise HTTPException(status_code=404, detail=f"File tidak ditemukan: {file_path}")
        if not file_path.lower().endswith('.pdf'):
            raise HTTPException(status_code=400, detail="File harus PDF")


        # ── Tutup koneksi ChromaDB lama ────────────────────────────────────────
        _release_chroma(self.vector_db)
        self.vector_db = None
        self.retriever = None


        # ── Backup DB lama sebelum dihapus ──────────────────────────────────────
        backup_dir = self.db_dir + "_backup"
        db_existed = os.path.exists(self.db_dir) and any(os.scandir(self.db_dir))
        if db_existed:
            if os.path.exists(backup_dir):
                _force_rmtree(backup_dir)
            shutil.copytree(self.db_dir, backup_dir)
            _force_rmtree(self.db_dir)
            logger.info("Database lama di-backup, siap diganti")


        # Pastikan folder target ada
        os.makedirs(self.db_dir, exist_ok=True)


        try:
            # STEP 1: Fast extraction dulu (10-50x lebih cepat dari hi_res)
            step_start = datetime.now()
            tesseract_ready = config.get("pdf.tesseract_available", False)
            base_languages = config.get("pdf.languages", ["ind", "eng"])
            ocr_lang = config.get("pdf.ocr_languages", "ind+eng")


            # Baca strategy dari config (default: hi_res)
            # Tesseract OCR fallback tetap pakai fast agar tidak hang.
            # hi_res: pakai ML layout detection, lebih akurat untuk soal gambar.
            strategy = config.get("pdf.strategy", "hi_res")


            loader_kwargs = {
                "file_path": file_path,
                "strategy": strategy,
                "mode": "elements",
                "extract_images_in_pdf": False,   # Skip di fase fast
                "infer_table_structure": False,    # Skip di fase fast
                "languages": base_languages,
            }


            loader = UnstructuredPDFLoader(**loader_kwargs)
            try:
                docs = loader.load()
            except Exception as load_err:
                if "tesseract" in str(load_err).lower():
                    logger.warning("Unstructured gagal (tesseract ref). Retry tanpa OCR.")
                    loader = UnstructuredPDFLoader(
                        file_path,
                        strategy="fast",
                        mode="elements",
                        extract_images_in_pdf=False,
                        infer_table_structure=False,
                        languages=base_languages,
                    )
                    docs = loader.load()
                else:
                    raise


            fast_elapsed = (datetime.now() - step_start).total_seconds()
            logger.info(f"[PERF] Fast extraction: {fast_elapsed:.1f}s -> {len(docs)} elemen")


            # Jika Unstructured gagal total, langsung coba simple extraction
            if not docs:
                logger.warning("Unstructured 0 elements, trying simple text extraction...")
                docs = self._simple_text_extraction(file_path)
                if docs:
                    logger.info(f"Simple extraction berhasil: {len(docs)} docs")


            # ── STEP 2: Cek kualitas — apakah cukup teks terambil? ────────
            total_chars = sum(len(d.page_content.strip()) for d in docs)
            file_size_kb = os.path.getsize(file_path) / 1024
            min_chars = config.get("pdf.ocr_min_chars_per_page", 50)
            use_ocr_fallback = config.get("pdf.ocr_fallback", True) and tesseract_ready


            is_likely_scanned = (
                (total_chars < min_chars * 5 and file_size_kb > 100)
                or total_chars < 100
            )


            if is_likely_scanned and use_ocr_fallback:
                logger.warning(
                    f"⚠️ PDF kemungkinan scan/gambar: hanya {total_chars} chars "
                    f"dari file {file_size_kb:.0f}KB. Mencoba OCR fallback..."
                )
                ocr_docs = self._ocr_fallback(file_path)
                if ocr_docs and sum(len(d.page_content) for d in ocr_docs) > total_chars:
                    docs = ocr_docs
                    logger.info(f"… OCR fallback menghasilkan {len(docs)} halaman "
                                f"({sum(len(d.page_content) for d in docs)} chars)")
                else:
                    logger.info("OCR fallback tidak lebih baik, pakai hasil Unstructured")
            elif not docs or total_chars == 0:
                logger.warning(f"Unstructured gagal: docs={len(docs)}, chars={total_chars}")


                # Coba OCR fallback jika tersedia
                if use_ocr_fallback:
                    logger.warning("Mencoba OCR fallback...")
                    docs = self._ocr_fallback(file_path)
                    if docs:
                        total_chars = sum(len(d.page_content.strip()) for d in docs)
                        logger.info(f"OCR fallback OK: {len(docs)} docs, {total_chars} chars")


                # Jika OCR gagal, coba simple text extraction (pdfplumber/PyPDF2)
                if not docs:
                    logger.warning("OCR gagal, mencoba simple text extraction...")
                    docs = self._simple_text_extraction(file_path)
                    if docs:
                        total_chars = sum(len(d.page_content.strip()) for d in docs)
                        logger.info(f"Simple extraction OK: {len(docs)} docs, {total_chars} chars")


                if not docs or total_chars == 0:
                    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
                    error_msg = (
                        f"❌ Gagal extract konten dari PDF: {os.path.basename(file_path)} ({file_size_mb:.2f}MB)\n"
                        f"\n📋 Yang sudah dicoba:\n"
                        f"  1. Unstructured PDF Loader (strategy='hi_res' or 'fast') → GAGAL\n"
                        f"  2. Simple text extraction (pypdf/pdfplumber/PyPDF2) → GAGAL\n"
                        f"  3. OCR Fallback (Tesseract) → TIDAK TERSEDIA atau GAGAL\n"
                        f"\n💡 Solusi:\n"
                        f"  • Pastikan file PDF adalah valid/tidak corrupted\n"
                        f"  • Jika PDF adalah scan/gambar:\n"
                        f"    - Install Tesseract-OCR: apt install tesseract-ocr tesseract-ocr-ind (Linux)\n"
                        f"    - atau gunakan setup dari DEPLOYMENT.md\n"
                        f"  • Jika PDF text-based:\n"
                        f"    - Update Requirements.txt dan run: pip install -r Requirements.txt\n"
                        f"    - Verifikasi: pip freeze | grep -E 'pypdf|pdfplumber|PyPDF2'\n"
                        f"  • Coba re-upload file PDF\n"
                    )
                    raise ValueError(error_msg)
            else:
                logger.info(f"PDF content OK: {total_chars} chars dari {len(docs)} elemen")


            chunks = self.text_splitter.split_documents(docs)
            logger.info(f"Dibuat {len(chunks)} semantic chunks")
            chunks = filter_complex_metadata(chunks)


            # ── Deteksi & tag metadata "bab" dari judul PDF ──────────────────
            # Pattern lebih fleksibel: juga cocokkan "Bab 1", "BAB I", "Unit 1",
            # "Tema 1", "Pelajaran 1", dll yang umum di buku SD
            BAB_PATTERN = re.compile(
                r'(?i)^(?:bab|chapter|bagian|unit|tema|pelajaran|topik)\s+(\w+)'
                r'|^(BAB|CHAPTER|UNIT|TEMA|PELAJARAN)\s+(\w+)'
            )
            current_bab = "0"
            for chunk in chunks:
                text    = chunk.page_content.strip()
                cat     = chunk.metadata.get("category", "")
                matched = BAB_PATTERN.match(text)
                # Cocokkan juga UncategorizedText & NarrativeText pendek (sering jadi judul)
                is_heading = cat in ("Title", "Header") or (cat in ("UncategorizedText", "NarrativeText") and len(text) < 80)
                if is_heading and matched:
                    bab_val = next(
                        (g for g in matched.groups() if g and g.lower() not in ("bab", "chapter", "bagian")),
                        None
                    )
                    if bab_val:
                        current_bab = bab_val.lower()
                        logger.info(f"Terdeteksi Bab: {current_bab} — '{text[:60]}'")
                chunk.metadata["bab"] = current_bab
            logger.info(f"Metadata 'bab' ditambahkan ke {len(chunks)} chunks")


            # FIX: Wrap Chroma init — 'RustBindingsAPI' bug in some chromadb 0.5.x
            # versions leaves a corrupted sqlite3. Delete and retry once on that error.
            try:
                self.vector_db = Chroma.from_documents(
                    documents=chunks,
                    embedding=self.embeddings,
                    persist_directory=self.db_dir
                )
            except Exception as _chroma_err:
                _err_str = str(_chroma_err)
                if "bindings" in _err_str or "RustBindings" in _err_str or "sqlite" in _err_str.lower():
                    logger.warning(f"ChromaDB init error ({_err_str[:80]}). Wiping sqlite3 and retrying...")
                    _sqlite = os.path.join(self.db_dir, "chroma.sqlite3")
                    if os.path.exists(_sqlite):
                        os.remove(_sqlite)
                    # Also remove uuid subdirs that ChromaDB may have partially created
                    for _entry in os.scandir(self.db_dir):
                        if _entry.is_dir():
                            _force_rmtree(_entry.path)
                    self.vector_db = Chroma.from_documents(
                        documents=chunks,
                        embedding=self.embeddings,
                        persist_directory=self.db_dir
                    )
                else:
                    raise


            self._build_rag_chain()
            self._build_quiz_chain()


            avg_size  = sum(len(c.page_content) for c in chunks) / len(chunks)
            proc_time = (datetime.now() - start_time).total_seconds()


            result = {
                "status":          "success",
                "total_elements":  len(docs),
                "total_chunks":    len(chunks),
                "avg_chunk_size":  round(avg_size, 2),
                "processing_time": round(proc_time, 2),
                "chains_built": {
                    "rag_chain":  self.rag_chain  is not None,
                    "quiz_chain": self.quiz_chain is not None
                },
                "ragas_available": self.ragas_evaluator.available,
                "timestamp":       datetime.now().isoformat()
            }
            logger.info(f"… Ingestion selesai: {result}")
            self._increment_stat("total_ingestions")
            if os.path.exists(backup_dir):
                _force_rmtree(backup_dir)
                logger.info("Backup DB lama dihapus (ingestion sukses)")
            return result


        except Exception as e:
            logger.error(f"Ingestion gagal: {e}", exc_info=True)
            if db_existed and os.path.exists(backup_dir):
                if os.path.exists(self.db_dir):
                    _force_rmtree(self.db_dir)
                shutil.copytree(backup_dir, self.db_dir)
                _force_rmtree(backup_dir)
                self.vector_db  = None
                self.rag_chain  = None
                self.quiz_chain = None
                logger.warning("⚠️ Ingestion gagal — DB lama dipulihkan dari backup")
            raise HTTPException(status_code=500, detail=f"Error ingestion: {str(e)}")


    # ─── ASK (GURU & SISWA) ──────────────────────────────────────


    @retry(stop=stop_after_attempt(2), wait=wait_exponential(min=1, max=4))
    def ask(
        self,
        question:           str,
        include_sources:    bool = True,
        include_validation: bool = True,
        run_ragas:          bool = False,
        ground_truth: Optional[str] = None
    ) -> Dict:
        start_time = datetime.now()
        self._increment_stat("total_queries")


        # ── Toxicity check via Llama Guard 3 ──────────────────────
        is_toxic, category, edu_message = self.toxicity_filter.check(question)
        if is_toxic:
            self._increment_stat("toxic_blocked")
            return {
                "answer":    edu_message,
                "is_toxic":  True,
                "category":  category,
                "is_valid":  False,
                "relevance_score": 0.0,
                "sources":   []
            }


        if not os.path.exists(self.db_dir):
            return {
                "answer": "❌ Buku belum diunggah. Silakan minta guru untuk mengunggah buku terlebih dahulu.",
                "is_valid": False, "sources": [], "relevance_score": 0.0
            }


        try:
            if self.rag_chain is None:
                self._build_rag_chain()
            if self.rag_chain is None:
                raise ValueError("Gagal membangun RAG chain")


            result = self.rag_chain.invoke(question)


            raw_answer = result["answer"]
            context    = result["context"]
            docs       = result["docs"]


            final_answer, guard_info = self.anti_hallucination.check_and_sanitize(
                raw_answer, context, question
            )
            logger.info(f"Guard: {guard_info}")


            relevance_result = self.relevance_scorer.calculate_combined_score(
                question, final_answer, context,
                methods=config.get("relevance.methods", ["cosine_similarity", "semantic_overlap"])
            )


            validation_result = self._validate_answer(final_answer, context, question) \
                                 if include_validation else None


            sources = []
            if include_sources and docs:
                for i, doc in enumerate(docs[:3]):
                    sources.append({
                        "index":    i + 1,
                        "content":  doc.page_content[:200] + "...",
                        "metadata": doc.metadata
                    })


            proc_time = (datetime.now() - start_time).total_seconds()


            if relevance_result["is_relevant"]:
                self._increment_stat("successful_answers")
            else:
                self._increment_stat("low_relevance_answers")


            ragas_result = {}
            if run_ragas and self.ragas_evaluator.available:
                ragas_result = self.ragas_evaluator.evaluate_single(
                    question=question,
                    answer=final_answer,
                    contexts=[doc.page_content for doc in docs],
                    ground_truth=ground_truth
                )


            self._save_evaluation(question, final_answer, relevance_result, context, proc_time)


            logger.info(
                f"… Q&A | Score: {relevance_result['final_score']} "
                f"| Guard: {guard_info['action']} | Time: {proc_time:.2f}s"
            )


            return {
                "answer":             final_answer,
                "is_toxic":           False,
                "is_valid":           relevance_result["is_relevant"],
                "relevance_score":    relevance_result["final_score"],
                "relevance_details":  relevance_result.get("method_scores", {}),
                "anti_hallucination": guard_info,
                "validation":         validation_result,
                "sources":            sources,
                "metrics": {
                    "processing_time": round(proc_time, 3),
                    "docs_retrieved":  len(docs),
                    "context_length":  len(context)
                },
                "ragas":     ragas_result,
                "timestamp": datetime.now().isoformat()
            }


        except Exception as e:
            self._increment_stat("failed_answers")
            logger.error(f"Error ask(): {e}", exc_info=True)
            return {
                "answer": f"❌ Terjadi kesalahan: {str(e)}",
                "is_valid": False, "sources": [], "relevance_score": 0.0, "error": str(e)
            }


    # ─── GENERATE QUIZ (GURU & SISWA) ────────────────────────────


    def generate_quiz(
        self,
        num_questions: int = None,
        bab: Optional[Any] = None
    ) -> Dict:
        """
        Generate soal pilihan ganda dari materi buku.


        Parameters
        ----------
        num_questions : int
            Jumlah soal (1-10, default 3).
        bab : str | int | list | None
            Filter chapter ChromaDB.
        """
        if num_questions is None:
            num_questions = config.get("quiz.default_questions", 3)


        max_q = config.get("quiz.max_questions", 10)
        if not (1 <= num_questions <= max_q):
            return {"status": "error", "message": f"Jumlah soal harus 1-{max_q}", "quiz": []}


        if not os.path.exists(self.db_dir):
            return {"status": "error",
                    "message": "Buku belum diunggah. Silakan minta guru untuk mengunggah buku.",
                    "quiz": []}


        try:
            if self.quiz_chain is None:
                self._build_quiz_chain()
            if self.quiz_chain is None:
                raise ValueError("Gagal membangun quiz chain")


            max_retry      = config.get("quiz.max_retry", 3)
            valid_questions: List[Dict] = []
            last_error     = ""


            for attempt in range(1, max_retry + 1):
                logger.info(f"Quiz attempt {attempt}/{max_retry} — target {num_questions} soal")
                request_count = min(num_questions + 2, max_q)


                try:
                    result = self.quiz_chain.invoke(
                        {"num_questions": request_count, "bab": bab}
                    )
                except ValueError as ve:
                    return {"status": "error", "message": str(ve), "quiz": []}


                questions_raw = result.get("questions", [])
                logger.info(f"LLM hasilkan {len(questions_raw)} soal mentah")


                context_for_validation = getattr(self, '_last_quiz_context', '')


                for i, q in enumerate(questions_raw):
                    fixed = self.quiz_validator.validate_and_fix(q, context=context_for_validation)
                    if fixed:
                        try:
                            validated = QuizQuestion(**fixed)
                            # Kompatibel Pydantic v1 dan v2
                            q_dict = (validated.model_dump()
                                      if hasattr(validated, 'model_dump')
                                      else validated.dict())
                            valid_questions.append(q_dict)
                            logger.info(f"Soal {i+1} valid")
                        except Exception as e:
                            logger.warning(f"Soal {i+1} Pydantic gagal: {e}")
                    else:
                        logger.warning(f"Soal {i+1} tidak valid")


                # Deduplikasi berdasarkan teks soal
                seen, unique = set(), []
                for q in valid_questions:
                    key = q["soal"][:80]
                    if key not in seen:
                        seen.add(key); unique.append(q)
                valid_questions = unique


                logger.info(f"Total soal valid: {len(valid_questions)}/{num_questions}")
                if len(valid_questions) >= num_questions:
                    break
                last_error = (f"Percobaan {attempt}: soal valid hanya "
                              f"{len(valid_questions)}/{num_questions}.")


            if len(valid_questions) > num_questions:
                valid_questions = random.sample(valid_questions, num_questions)


            if not valid_questions:
                return {
                    "status":  "error",
                    "message": f"Tidak ada soal valid setelah {max_retry}x percobaan. {last_error}",
                    "quiz":    []
                }


            return {
                "status":     "success",
                "quiz":       valid_questions,
                "total":      len(valid_questions),
                "requested":  num_questions,
                "bab_filter": bab,
                "timestamp":  datetime.now().isoformat()
            }


        except Exception as e:
            logger.error(f"Quiz generation error: {e}", exc_info=True)
            return {"status": "error", "message": f"Gagal buat quiz: {str(e)}", "quiz": []}




    def _validate_answer(self, answer: str, context: str, question: str) -> Dict:
        issues, suggestions = [], []


        if len(answer.strip()) < 10:
            issues.append("Jawaban terlalu pendek")


        word_count = len(answer.split())
        if word_count > 500:
            issues.append(f"Jawaban terlalu panjang ({word_count} kata)")
            suggestions.append("Ringkas jawaban")


        overlap    = self.anti_hallucination.compute_token_overlap(answer, context)
        is_no_info = self.anti_hallucination.is_valid_no_info(answer)


        min_overlap = config.get("anti_hallucination.min_context_overlap", 0.20)
        if overlap < min_overlap and not is_no_info:
            issues.append(f"Context overlap rendah ({overlap:.2f})")
            suggestions.append("Pastikan jawaban dari materi")


        if self.anti_hallucination.has_hallucination_signal(answer):
            issues.append("Terdeteksi penggunaan pengetahuan di luar materi")
            suggestions.append("Hapus referensi ke sumber eksternal")


        return {
            "is_valid":        len(issues) == 0,
            "confidence":      round(max(0.0, min(1.0, 1.0 - len(issues) * 0.2)), 3),
            "issues":          issues,
            "suggestions":     suggestions,
            "word_count":      word_count,
            "context_overlap": overlap
        }


    def _save_evaluation(self, question, answer, relevance_result, context, proc_time):
        try:
            eval_file = os.path.join(self.eval_dir, "qa_evaluation.jsonl")
            data = {
                "timestamp":       datetime.now().isoformat(),
                "question":        question,
                "answer":          answer,
                "relevance_score": relevance_result["final_score"],
                "method_scores":   relevance_result.get("method_scores", {}),
                "is_relevant":     relevance_result["is_relevant"],
                "processing_time": proc_time,
                "context_preview": context[:300],
                "answer_length":   len(answer),
                "context_length":  len(context)
            }
            with open(eval_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(data, ensure_ascii=False) + "\n")
        except Exception as e:
            logger.error(f"Gagal simpan evaluasi: {e}")


    def get_statistics(self) -> Dict:
        try:
            eval_file   = os.path.join(self.eval_dir, "qa_evaluation.jsonl")
            rel_scores, proc_times = [], []
            if os.path.exists(eval_file):
                with open(eval_file, "r", encoding="utf-8") as f:
                    for line in f:
                        try:
                            d = json.loads(line)
                            rel_scores.append(d.get("relevance_score", 0))
                            proc_times.append(d.get("processing_time", 0))
                        except Exception:
                            continue


            with self._stats_lock:
                stats_snapshot = dict(self.stats)


            return {
                "total_queries":          stats_snapshot.get("total_queries", 0),
                "successful_answers":     stats_snapshot.get("successful_answers", 0),
                "low_relevance_answers":  stats_snapshot.get("low_relevance_answers", 0),
                "failed_answers":         stats_snapshot.get("failed_answers", 0),
                "toxic_blocked":          stats_snapshot.get("toxic_blocked", 0),
                "average_relevance_score":round(np.mean(rel_scores),   3) if rel_scores  else 0,
                "median_relevance_score": round(np.median(rel_scores),  3) if rel_scores  else 0,
                "average_processing_time":round(np.mean(proc_times),   3) if proc_times  else 0,
                "total_evaluations":      len(rel_scores),
                "database_exists":        os.path.exists(self.db_dir),
                "chains_ready": {
                    "rag_chain":  self.rag_chain  is not None,
                    "quiz_chain": self.quiz_chain is not None
                },
                "toxicity_filter": {
                    "model":   config.get("llm.llama_guard_model", "llama-guard3:8b"),
                    "enabled": self.toxicity_filter.enabled
                },
                "ragas_evaluation": self.ragas_evaluator.get_summary()
            }
        except Exception as e:
            logger.error(f"Error get_statistics: {e}")
            with self._stats_lock:
                return dict(self.stats)




# ========================================
# INITIALIZE (via lifespan — bukan module level)
# ========================================
# Deklarasi global agar semua endpoint bisa mengakses setelah startup.
# Inisialisasi dilakukan di dalam lifespan sehingga tidak otomatis berjalan
# saat modul di-import (misal saat testing), melainkan hanya saat server start.


assistant: Optional[UltimateAssistantLCEL] = None
assistant_init_error: Optional[str] = None




@asynccontextmanager
async def lifespan(app: FastAPI):
    """
    FastAPI lifespan event.
    - startup : inisialisasi assistant (load embeddings, LLM, ChromaDB)
    - shutdown: placeholder untuk cleanup resources jika diperlukan


    FIX KRITIS: Lifespan HARUS yield secepatnya agar server mulai menerima
    koneksi. Model loading dilakukan di background thread. Endpoint ringan
    seperti /auth/verify, /health langsung responsif. Endpoint berat
    (/chat, /quiz) return 503 sampai assistant siap.
    """
    global assistant, assistant_init_error
    logger.info("=" * 60)
    logger.info("Startup: server akan menerima koneksi segera...")
    logger.info("         Model loading berjalan di background thread.")
    logger.info("=" * 60)


    def _init_assistant_background():
        """Heavy initialization in background thread."""
        global assistant, assistant_init_error
        try:
            logger.info("[BG-INIT] Mulai load UltimateAssistantLCEL...")
            assistant = UltimateAssistantLCEL()
            assistant_init_error = None
            logger.info("[BG-INIT] ✓ Assistant siap digunakan!")
        except Exception as e:
            assistant = None
            assistant_init_error = str(e)
            logger.error(f"[BG-INIT] ❌ Gagal: {e}", exc_info=True)


    # Start in background — do NOT await/join
    _init_thread = threading.Thread(target=_init_assistant_background, daemon=True)
    _init_thread.start()


    yield  # Server immediately starts accepting connections


    # --- shutdown ---
    logger.info("Shutdown: membersihkan resources...")
    assistant = None
    assistant_init_error = None


# ========================================
# FASTAPI APP
# ========================================


app = FastAPI(
    title="Asisten Belajar SD",
    version="7.2",
    description=(
        "RAG + Llama Guard 3 Toxicity Filter + RAGAS Lokal (Ollama) "
        "+ Anti-Hallucination + Quiz Generation\n\n"
        "**Autentikasi**: sertakan header `X-Auth-Token` pada setiap request.\n"
        "- Token **guru**  → akses penuh (upload + chat + quiz)\n"
        "- Token **siswa** → hanya chat dan quiz"
    ),
    lifespan=lifespan,
)


app.add_middleware(
    CORSMiddleware,
    allow_origins=config.get("api.cors_origins", ["http://localhost:3000"]),
    allow_credentials=True,
    allow_methods=["GET", "POST", "DELETE", "OPTIONS", "PUT", "PATCH"],
    allow_headers=["Content-Type", "X-Auth-Token", "ngrok-skip-browser-warning"],
)




# ========================================
# ENDPOINTS
# ========================================




@app.get("/")
async def root():
    return {
        "status":  "running" if assistant else "degraded",
        "version": "7.2",
        "assistant_ready": assistant is not None,


        "init_error": assistant_init_error,


        "ragas_available":      RAGAS_AVAILABLE,
        "llm_model":            config.get("llm.primary_model"),
        "llama_guard_model":    config.get("llm.llama_guard_model"),
        "llama_guard_enabled":  assistant.toxicity_filter.enabled if assistant else False,
        "docs_url":             "/docs",
        "auth_header":          "X-Auth-Token",
        "roles": {
            "guru":  "upload + chat + quiz + statistics + evaluate + config + reset",
            "siswa": "chat + quiz"
        }
    }






@app.get("/health", summary="[PUBLIC] Health check", tags=["public"])
async def health():
    """Health check - cek apakah assistant sudah siap."""
    return {
        "status": "ok" if assistant else "degraded",
        "assistant_ready": assistant is not None,
        "init_error": assistant_init_error,
        "version": "7.2"
    }


# == AUTH VERIFY == VALIDASI TOKEN + ROLE ==============================


@app.post("/auth/verify", summary="[PUBLIC] Validasi token dan role", tags=["public"])
async def verify_auth(request: Request):
    """
    Endpoint untuk validasi token saat login.
    Frontend kirim {"token": "xxx", "role": "guru/siswa"} dan backend cek apakah cocok.
    Mencegah login guru pakai token siswa dan sebaliknya.
    """
    try:
        body = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Body harus JSON")


    token = body.get("token", "").strip()
    requested_role = body.get("role", "").strip().lower()


    if not token:
        raise HTTPException(status_code=401, detail="Token tidak boleh kosong")
    if requested_role not in ("guru", "siswa"):
        raise HTTPException(status_code=400, detail="Role harus 'guru' atau 'siswa'")


    # Resolve actual tokens
    guru_token  = os.environ.get("GURU_TOKEN")  or config.get("auth.guru_token",  "")
    siswa_token = os.environ.get("SISWA_TOKEN") or config.get("auth.siswa_token", "")
    if not guru_token or "GANTI" in guru_token:
        guru_token = os.environ.get("DEV_GURU_TOKEN", "Guru2026")
    if not siswa_token or "GANTI" in siswa_token:
        siswa_token = os.environ.get("DEV_SISWA_TOKEN", "Siswa2026")


    # Determine which role the token actually belongs to
    actual_role = None
    if guru_token and secrets.compare_digest(token.encode(), guru_token.encode()):
        actual_role = "guru"
    elif siswa_token and secrets.compare_digest(token.encode(), siswa_token.encode()):
        actual_role = "siswa"


    if actual_role is None:
        logger.warning(f"Auth verify failed: invalid token")
        return {"valid": False, "message": "Token tidak valid. Hubungi administrator."}


    if actual_role != requested_role:
        logger.warning(f"Auth verify: token milik '{actual_role}' tapi login sebagai '{requested_role}'")
        return {
            "valid": False,
            "message": f"Token ini milik {actual_role}, bukan {requested_role}. Pilih role yang benar."
        }


    logger.info(f"Auth verify: role={actual_role} OK")
    return {"valid": True, "role": actual_role, "message": "Login berhasil"}








# ── PUBLIC — CEK STATUS BUKU ─────────────────────────────────────────────


@app.get("/status", summary="[PUBLIC] Cek apakah buku sudah diunggah", tags=["public"])
async def book_status():
    """
    Endpoint PUBLIC — Tidak memerlukan autentikasi X-Auth-Token.
    Digunakan oleh frontend untuk polling status buku tanpa blocker proxy/middleware.
    """
    if assistant is None:
        return {
            "book_uploaded": False,
            "chain_ready": False,
            "message": "⏳ Server sedang inisialisasi. Coba lagi beberapa detik."
        }


    db_ready    = os.path.exists(assistant.db_dir) and any(
        f for f in os.listdir(assistant.db_dir) if not f.startswith(".")
    ) if os.path.exists(assistant.db_dir) else False
    chain_ready = assistant.rag_chain is not None
    return {
        "book_uploaded":  db_ready,
        "chain_ready":    chain_ready,
        "message": (
            "… Buku sudah tersedia. Kamu bisa mulai bertanya!"
            if db_ready else
            "❌ Buku belum diunggah. Silakan tunggu guru mengunggah buku."
        )
    }




# ── GURU ONLY ────────────────────────────────────────────────────────────────



# ── Book Registry ─────────────────────────────────────────────────────────────
import json as _json_mod, re as _re_mod

_BOOK_REGISTRY_PATH = "./book_registry.json"
_book_registry_lock = threading.Lock()
_assistant_chain_lock = threading.RLock()  # protects assistant state during swaps


def _load_book_registry() -> list:
    with _book_registry_lock:
        if os.path.exists(_BOOK_REGISTRY_PATH):
            try:
                with open(_BOOK_REGISTRY_PATH, "r", encoding="utf-8") as _f:
                    return _json_mod.load(_f)
            except Exception:
                pass
        return []


def _save_book_registry(books: list):
    with _book_registry_lock:
        with open(_BOOK_REGISTRY_PATH, "w", encoding="utf-8") as _f:
            _json_mod.dump(books, _f, ensure_ascii=False, indent=2)


def _make_book_id(filename: str) -> str:
    base = filename[:-4] if filename.lower().endswith(".pdf") else filename
    slug = _re_mod.sub(r"[^a-z0-9]+", "_", base.lower())[:30].strip("_")
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{slug}_{ts}"


def _books_base_dir() -> str:
    vector_db = config.get("paths.vector_db", "./vector_db_lcel")
    parent    = os.path.dirname(os.path.abspath(vector_db))
    return os.path.join(parent, "vector_db_books")


# ── Job tracking untuk async upload ──────────────────────────────────────────
_upload_jobs: Dict[str, Dict] = {}
_upload_jobs_lock = threading.Lock()

# -- Job tracking untuk async chat --
_chat_jobs: Dict[str, Dict] = {}
_chat_jobs_lock = threading.Lock()




@app.post("/upload", summary="[GURU] Upload PDF buku pelajaran")
async def upload(
    file: UploadFile = File(...),
    role: str = Depends(require_guru),
    _rl:  None = Depends(rate_limit)
):
    if not file.filename.lower().endswith('.pdf'):
        raise HTTPException(status_code=400, detail="File harus berformat PDF")


    content = await file.read()


    max_mb    = config.get("pdf.max_size_mb", 50)
    max_bytes = max_mb * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File terlalu besar. Maksimum {max_mb} MB."
        )


    if not content.startswith(b'%PDF'):
        raise HTTPException(
            status_code=400,
            detail="File bukan PDF asli (magic bytes tidak valid)."
        )


    logger.info(f"[GURU] Upload: {file.filename} ({len(content) / 1024:.1f} KB)")
    temp_dir = config.get("paths.temp", "./temp")
    os.makedirs(temp_dir, exist_ok=True)


    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False, dir=temp_dir)
    tmp.write(content)
    tmp.close()
    tmp_path = tmp.name


    job_id = secrets.token_hex(8)
    with _upload_jobs_lock:
        _upload_jobs[job_id] = {"status": "processing", "result": None, "error": None}


    book_id     = _make_book_id(file.filename)
    book_db_dir = os.path.join(_books_base_dir(), book_id)
    os.makedirs(book_db_dir, exist_ok=True)


    def _ingest_job(path: str, jid: str, orig_name: str, bid: str, bdir: str):
        try:
            # Use isolated ingest: does NOT touch current assistant state
            # Chat/quiz keep working with the old book during this
            result = assistant._ingest_isolated(path, bdir)
            new_vector_db = result.pop("_new_vector_db", None)

            # ── Atomic swap: switch assistant to the new book ──────────
            with _assistant_chain_lock:
                old_vdb = assistant.vector_db
                assistant.vector_db  = new_vector_db
                assistant.retriever  = None
                assistant.rag_chain  = None
                assistant.quiz_chain = None
                assistant.db_dir     = bdir
                assistant._build_rag_chain()
                assistant._build_quiz_chain()

            # Release old ChromaDB AFTER new chains are ready
            if old_vdb is not None:
                try:
                    _release_chroma(old_vdb)
                except Exception:
                    pass

            # Register book in registry
            books = _load_book_registry()
            books = [b for b in books if b["book_id"] != bid]
            for b in books:
                b["is_active"] = False
            books.append({
                "book_id":         bid,
                "filename":        orig_name,
                "uploaded_at":     datetime.now().isoformat(),
                "db_dir":          bdir,
                "total_chunks":    result.get("total_chunks", 0),
                "total_elements":  result.get("total_elements", 0),
                "avg_chunk_size":  round(result.get("avg_chunk_size", 0), 1),
                "processing_time": round(result.get("processing_time", 0), 1),
                "is_active":       True,
            })
            _save_book_registry(books)
            with _upload_jobs_lock:
                _upload_jobs[jid] = {"status": "done", "result": result, "error": None}
            logger.info(f"[UPLOAD JOB {jid}] Success: {orig_name} -> {bid}")
        except Exception as exc:
            logger.error(f"[UPLOAD JOB {jid}] Error: {exc}")
            with _upload_jobs_lock:
                _upload_jobs[jid] = {"status": "error", "result": None, "error": str(exc)}
        finally:
            if os.path.exists(path):
                os.remove(path)


    thread = threading.Thread(
        target=_ingest_job,
        args=(tmp_path, job_id, file.filename, book_id, book_db_dir),
        daemon=True
    )
    thread.start()


    return {
        "job_id":  job_id,
        "status":  "processing",
        "message": "Upload diterima, proses berjalan di background",
        "book_id": book_id,
    }



@app.get("/upload/status/{job_id}", summary="[GURU] Cek status upload")
async def upload_status(job_id: str, role: str = Depends(require_guru)):
    with _upload_jobs_lock:
        job = _upload_jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="Job tidak ditemukan")
    return {"job_id": job_id, **job}




@app.get("/books", summary="[GURU] Daftar semua buku yang pernah diupload")
async def list_books(role: str = Depends(require_guru)):
    """Mengembalikan daftar buku dari registry beserta status aktif/tidak."""
    books = _load_book_registry()

    # Auto-migrasi: jika registry kosong tapi assistant sudah punya DB aktif,
    # daftarkan otomatis agar riwayat tidak tampak kosong.
    if not books and assistant is not None:
        db_dir = assistant.db_dir
        has_data = False
        if os.path.exists(db_dir):
            try:
                has_data = any(e for e in os.scandir(db_dir) if not e.name.startswith("."))
            except Exception:
                pass
        if has_data:
            display_name = (
                os.path.basename(os.path.normpath(db_dir))
                if db_dir not in ("./vector_db_lcel", "vector_db_lcel")
                else "Buku yang sudah ada (sebelum fitur riwayat)"
            )
            books = [{
                "book_id":        "buku_lama",
                "filename":       display_name,
                "uploaded_at":    datetime.now().isoformat(),
                "db_dir":         db_dir,
                "total_chunks":   0,
                "total_elements": 0,
                "is_active":      True,
            }]
            _save_book_registry(books)
            logger.info(f"[MIGRATION] Auto-registered existing book: {db_dir}")

    return {"books": books, "total": len(books)}


@app.post("/books/{book_id}/activate", summary="[GURU] Aktifkan buku tertentu")
async def activate_book(book_id: str, role: str = Depends(require_guru)):
    """
    Ganti buku yang aktif tanpa perlu re-upload.
    Mengaktifkan ChromaDB dari direktori buku yang dipilih.
    """
    if assistant is None:
        raise HTTPException(status_code=503, detail="Server belum siap.")
    books = _load_book_registry()
    target = next((b for b in books if b["book_id"] == book_id), None)
    if target is None:
        raise HTTPException(status_code=404, detail="Buku tidak ditemukan dalam registry.")
    bdir = target["db_dir"]
    if not os.path.exists(bdir) or not any(
        e for e in os.scandir(bdir) if not e.name.startswith(".")
    ):
        raise HTTPException(
            status_code=400,
            detail="Data buku tidak ditemukan di server. Silakan upload ulang."
        )
    try:
        with _assistant_chain_lock:
            old_vdb = assistant.vector_db
            # Build new state first
            assistant.vector_db  = None
            assistant.retriever  = None
            assistant.rag_chain  = None
            assistant.quiz_chain = None
            assistant.db_dir     = bdir
            assistant._build_rag_chain()
            assistant._build_quiz_chain()
        # Release old AFTER new chains are ready
        if old_vdb is not None:
            try:
                _release_chroma(old_vdb)
            except Exception:
                pass
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Gagal mengaktifkan buku: {str(e)}")
    for b in books:
        b["is_active"] = (b["book_id"] == book_id)
    _save_book_registry(books)
    logger.info(f"[GURU] Activated book: {target['filename']} ({book_id})")
    return {"status": "ok", "message": f"Buku '{target['filename']}' sekarang aktif."}


@app.delete("/books/{book_id}", summary="[GURU] Hapus buku dari history")
async def delete_book(book_id: str, role: str = Depends(require_guru)):
    """
    Hapus buku dari registry dan hapus data ChromaDB-nya.
    Buku yang sedang aktif tidak bisa dihapus.
    """
    books = _load_book_registry()
    target = next((b for b in books if b["book_id"] == book_id), None)
    if target is None:
        raise HTTPException(status_code=404, detail="Buku tidak ditemukan dalam registry.")
    if target.get("is_active"):
        raise HTTPException(
            status_code=400,
            detail="Tidak bisa menghapus buku yang sedang aktif. Aktifkan buku lain terlebih dahulu."
        )
    bdir = target["db_dir"]
    if os.path.exists(bdir):
        _force_rmtree(bdir)
    books = [b for b in books if b["book_id"] != book_id]
    _save_book_registry(books)
    logger.info(f"[GURU] Deleted book: {target['filename']} ({book_id})")
    return {"status": "ok", "message": f"Buku '{target['filename']}' berhasil dihapus."}

@app.post("/evaluate/ragas", summary="[GURU] Batch evaluasi RAGAS")
async def evaluate_ragas(
    req: RAGASEvaluationRequest,
    role: str = Depends(require_guru),
    _rl:  None = Depends(rate_limit)
):
    """
    Batch RAGAS evaluation menggunakan Ollama lokal (tanpa OpenAI).
    **Hanya guru** yang dapat menjalankan evaluasi ini.


    ⚠️ WARNING: RAGAS processing SANGAT LAMBAT (2-5 menit per 3 questions).
    Production: pastikan nginx/reverse proxy timeout ≥ 600 detik.
    """
    if not RAGAS_AVAILABLE:
        raise HTTPException(
            status_code=503,
            detail="RAGAS tidak tersedia. Install: pip install ragas datasets"
        )
    if not os.path.exists(assistant.db_dir):
        raise HTTPException(status_code=400, detail="Buku belum diunggah")


    if not req.questions or len(req.questions) == 0:
        raise HTTPException(status_code=400, detail="Minimal 1 pertanyaan diperlukan")


    max_questions = 5
    if len(req.questions) > max_questions:
        raise HTTPException(
            status_code=400,
            detail=f"Maksimal {max_questions} pertanyaan sekaligus (RAGAS sangat lambat). "
                   f"Anda memasukkan {len(req.questions)} pertanyaan."
        )


    ground_truths = req.ground_truths
    if ground_truths and len(ground_truths) != len(req.questions):
        raise HTTPException(
            status_code=400,
            detail="Jumlah ground_truths harus sama dengan questions"
        )


    # Pre-check: Ollama connectivity
    try:
        test_response = assistant.llm_qa.invoke("test")
        logger.info("✓ Pre-check: Ollama responding OK")
    except Exception as e:
        logger.error(f"❌ Pre-check: Ollama tidak merespons: {e}")
        raise HTTPException(
            status_code=503,
            detail=f"Ollama service tidak merespons. Error: {str(e)[:100]}. "
                   f"Pastikan: ollama serve berjalan di {config.get('llm.ollama_base_url')}"
        )


    def _run_batch_sync():
        try:
            start_time = datetime.now()
            logger.info(f"[RAGAS] Mulai evaluasi batch: {len(req.questions)} questions")


            answers, contexts = [], []
            for i, q in enumerate(req.questions):
                logger.info(f"[RAGAS] [{i+1}/{len(req.questions)}] Processing: {q[:60]}...")
                try:
                    with _assistant_chain_lock:
                        res = assistant.ask(q, include_sources=True, include_validation=False)
                    answers.append(res.get("answer", ""))
                    raw_ctx = [s["content"].replace("...", "") for s in res.get("sources", [])]
                    contexts.append(raw_ctx if raw_ctx else [""])
                except Exception as ask_err:
                    err_str = str(ask_err)
                    if "bindings" in err_str.lower() or "rustbindings" in err_str.lower():
                        raise ValueError("Sedang terjadi pergantian buku. Silakan coba lagi.")
                    logger.error(f"[RAGAS] Error ask question {i+1}: {ask_err}")
                    raise


            logger.info(f"[RAGAS] Answer collection selesai, now evaluating...")
            with _assistant_chain_lock:
                result = assistant.ragas_evaluator.evaluate_batch(
                    questions=req.questions,
                    answers=answers,
                    contexts=contexts,
                    ground_truths=ground_truths
                )


            duration = (datetime.now() - start_time).total_seconds()
            logger.info(f"[RAGAS] ✓ Batch evaluation selesai ({duration:.1f}s)")
            result["duration_seconds"] = duration
            return result


        except Exception as e:
            logger.error(f"[RAGAS] ❌ Batch evaluation gagal: {e}", exc_info=True)
            raise


    try:
        # Set timeout 600 detik (10 menit max untuk batch RAGAS)
        return await asyncio.wait_for(
            asyncio.to_thread(_run_batch_sync),
            timeout=600.0
        )
    except asyncio.TimeoutError:
        logger.error("[RAGAS] ❌ Batch evaluation timeout (600s)")
        raise HTTPException(
            status_code=504,
            detail="RAGAS evaluation timeout (>10 menit). "
                   "Production: pastikan nginx timeout ≥ 600s. "
                   "Reduce jumlah questions atau optimize Ollama settings."
        )
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"[RAGAS] ❌ Unexpected error: {e}", exc_info=True)
        raise HTTPException(
            status_code=500,
            detail=f"RAGAS evaluation error: {str(e)[:200]}"
        )




@app.get("/evaluate/ragas/summary", summary="[GURU] Ringkasan evaluasi RAGAS")
async def ragas_summary(role: str = Depends(require_guru)):
    """Ringkasan semua evaluasi RAGAS yang pernah dijalankan. **Hanya guru.**"""
    def _summary_with_lock():
        with _assistant_chain_lock:
            return assistant.ragas_evaluator.get_summary()
    return await asyncio.to_thread(_summary_with_lock)




@app.get("/statistics", summary="[GURU] Statistik sistem")
async def statistics(role: str = Depends(require_guru)):
    """Statistik sistem lengkap + ringkasan RAGAS. **Hanya guru.**"""
    def _stats_with_lock():
        with _assistant_chain_lock:
            return assistant.get_statistics()
    return await asyncio.to_thread(_stats_with_lock)




@app.get("/config", summary="[GURU] Konfigurasi sistem")
async def get_config(role: str = Depends(require_guru)):
    """Tampilkan konfigurasi sistem aktif. **Hanya guru.**"""
    return {
        "config_file":        config.config_path,
        "paths":              config.config.get("paths"),
        "llm":                config.config.get("llm"),
        "embedding":          config.config.get("embedding"),
        "pdf":                config.config.get("pdf"),
        "retrieval":          config.config.get("retrieval"),
        "relevance":          config.config.get("relevance"),
        "quiz":               config.config.get("quiz"),
        "anti_hallucination": config.config.get("anti_hallucination"),
        "ragas":              config.config.get("ragas")
    }




@app.delete("/reset", summary="[GURU] Reset semua database & buku")
async def reset_database(role: str = Depends(require_guru)):
    """Reset SEMUA database, chains, dan riwayat buku. **Hanya guru.**"""
    try:
        with _assistant_chain_lock:
            # Tutup koneksi ChromaDB dulu
            _release_chroma(assistant.vector_db)
            assistant.vector_db  = None
            assistant.retriever  = None
            assistant.rag_chain  = None
            assistant.quiz_chain = None

            # Hapus folder DB aktif
            if os.path.exists(assistant.db_dir):
                _force_rmtree(assistant.db_dir)
            os.makedirs(assistant.db_dir, exist_ok=True)

            # Hapus SEMUA folder buku di ./vector_db_books/
            books_base = _books_base_dir()
            if os.path.exists(books_base):
                _force_rmtree(books_base)
            os.makedirs(books_base, exist_ok=True)

            # Hapus juga vector_db_lcel jika bukan db_dir aktif
            default_db = "./vector_db_lcel"
            if os.path.abspath(default_db) != os.path.abspath(assistant.db_dir):
                if os.path.exists(default_db):
                    _force_rmtree(default_db)

            # Reset book registry
            _save_book_registry([])

            # Reset assistant db_dir to default
            assistant.db_dir = default_db
            os.makedirs(assistant.db_dir, exist_ok=True)

        logger.info("[GURU] SEMUA database, buku, dan chains direset")
        return {"status": "success", "message": "Semua database dan riwayat buku berhasil direset."}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))




# ── GURU & SISWA ─────────────────────────────────────────────────────────────


@app.post("/chat", summary="[GURU & SISWA] Tanya jawab dengan buku")
async def chat(
    req:  ChatRequest,
    role: str  = Depends(require_any),
    _rl:  None = Depends(rate_limit)
):
    """
    Chat Q&A -- dikembalikan async via job_id.
    Gunakan GET /chat/status/{job_id} untuk mengambil hasilnya.
    """
    if assistant is None:
        detail = "Assistant belum siap. Cek log startup dan coba lagi."
        if assistant_init_error:
            detail = f"Assistant gagal inisialisasi: {assistant_init_error}"
        raise HTTPException(status_code=503, detail=detail)


    logger.info(f"[{role.upper()}] Chat: {req.prompt[:60]}...")

    job_id = secrets.token_hex(8)
    with _chat_jobs_lock:
        _chat_jobs[job_id] = {"status": "processing", "result": None, "error": None}

    def _chat_job(jid: str):
        try:
            with _assistant_chain_lock:
                result = assistant.ask(
                    req.prompt,
                    req.include_sources,
                    req.include_validation,
                    run_ragas=req.run_ragas,
                    ground_truth=req.ground_truth
                )
            with _chat_jobs_lock:
                _chat_jobs[jid] = {"status": "done", "result": result, "error": None}
        except Exception as exc:
            err_msg = str(exc)
            # Friendly message for ChromaDB internal errors
            if "bindings" in err_msg.lower() or "rustbindings" in err_msg.lower():
                err_msg = "Sedang terjadi pergantian buku. Silakan coba lagi dalam beberapa detik."
            logger.error(f"[CHAT JOB {jid}] Error: {exc}")
            with _chat_jobs_lock:
                _chat_jobs[jid] = {"status": "error", "result": None, "error": err_msg}

    thread = threading.Thread(target=_chat_job, args=(job_id,), daemon=True)
    thread.start()

    return {"job_id": job_id, "status": "processing", "message": "Permintaan diterima, sedang diproses"}




@app.get("/chat/status/{job_id}", summary="[GURU & SISWA] Cek status chat")
async def chat_job_status(job_id: str, role: str = Depends(require_any)):
    with _chat_jobs_lock:
        job = _chat_jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="Job tidak ditemukan")
    return {"job_id": job_id, **job}
@app.get("/quiz", summary="[GURU & SISWA] Generate soal kuis")
async def quiz(
    n:    int = None,
    bab:  Optional[str] = None,
    role: str  = Depends(require_any),
    _rl:  None = Depends(rate_limit)
):
    """
    Generate soal kuis pilihan ganda dari materi buku.


    **Parameter:**
    - `n`   : jumlah soal (default 3, maks 10)
    - `bab` : filter chapter (opsional). Contoh: `?bab=2` atau `?bab=1,2`.
    """
    if assistant is None:
        detail = "Assistant belum siap. Cek log startup dan coba lagi."
        if assistant_init_error:
            detail = f"Assistant gagal inisialisasi: {assistant_init_error}"
        raise HTTPException(status_code=503, detail=detail)


    if n is None:
        n = config.get("quiz.default_questions", 3)


    bab_parsed: Optional[Any] = None
    if bab:
        parts = [b.strip() for b in bab.split(",") if b.strip()]
        bab_parsed = parts if len(parts) > 1 else parts[0]


    logger.info(f"[{role.upper()}] Quiz: {n} soal, bab={bab_parsed}")
    # FIX: Jalankan di thread pool dengan chain lock
    def _quiz_with_lock():
        with _assistant_chain_lock:
            return assistant.generate_quiz(n, bab=bab_parsed)
    try:
        return await asyncio.to_thread(_quiz_with_lock)
    except Exception as exc:
        err_msg = str(exc)
        if "bindings" in err_msg.lower() or "rustbindings" in err_msg.lower():
            raise HTTPException(status_code=503, detail="Sedang terjadi pergantian buku. Silakan coba lagi.")
        raise HTTPException(status_code=500, detail=err_msg)




# ========================================
# MAIN
# ========================================


if __name__ == "__main__":
    import uvicorn


    guru_tok  = os.environ.get("GURU_TOKEN")  or config.get("auth.guru_token",  "")
    siswa_tok = os.environ.get("SISWA_TOKEN") or config.get("auth.siswa_token", "")


    if "GANTI" in guru_tok or "GANTI" in siswa_tok:
        logger.warning("=" * 70)
        logger.warning("⚠️  PERINGATAN KEAMANAN: Token masih menggunakan nilai default!")
        logger.warning("   Set env var sebelum production:")
        logger.warning("   export GURU_TOKEN=$(openssl rand -hex 32)")
        logger.warning("   export SISWA_TOKEN=$(openssl rand -hex 32)")
        logger.warning("=" * 70)


    logger.info("=" * 70)
    logger.info("ASISTEN BELAJAR SD v7.2")
    logger.info("=" * 70)
    logger.info(f"📁 Config          : {config.config_path}")
    logger.info(f"🤖 LLM             : {config.get('llm.primary_model')}")
    logger.info(f"🛡️  Toxicity Filter : Llama Guard 3 — {config.get('llm.llama_guard_model')}")
    logger.info(f"🔐 Role-Based Auth : header X-Auth-Token (token dari env var GURU_TOKEN / SISWA_TOKEN)")
    logger.info(f"🔍 Retrieval       : {config.get('retrieval.method')} "
                f"(k={config.get('retrieval.top_k')}, fetch_k={config.get('retrieval.fetch_k')})")
    logger.info(f"📊 RAGAS           : {'… Ollama Lokal' if RAGAS_AVAILABLE else '❌ Belum install'}")
    logger.info(f"🛡️  Anti-Halluc     : Strict={config.get('anti_hallucination.strict_mode')}")
    logger.info(f"⚡ Rate Limit      : {rate_limiter.max_requests} req/{rate_limiter.window}s per token")
    logger.info(f"🌐 URL             : http://localhost:{config.get('api.port', 8000)}")
    logger.info(f"📖 API Docs        : http://localhost:{config.get('api.port', 8000)}/docs")
    # FIX: Token TIDAK dicetak ke log
    logger.info("=" * 70)


    # Detect if running in Jupyter/Colab
    is_jupyter = False
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            is_jupyter = True
            logger.info("... Jupyter/Colab detected – running uvicorn in background thread")
    except (ImportError, NameError):
        pass


    if is_jupyter:
        # Run uvicorn in a background thread with its own asyncio event loop.
        # Colab re-runs: kill any existing process on the port first so the
        # new uvicorn can bind cleanly.
        import uvicorn
        import threading
        import subprocess as _sp


        _host = config.get("api.host", "0.0.0.0")
        _port = config.get("api.port", 8000)


        # --- Kill any existing server on this port (handles Cell 8 re-runs) ---
        try:
            _sp.run(["fuser", "-k", f"{_port}/tcp"], capture_output=True)
            logger.info(f"... Released port {_port} (killed existing process if any)")
            time.sleep(2)  # give OS time to free the port
        except Exception:
            pass  # fuser not available on all systems – ignore


        def _run_server():
            # Use asyncio.run() + uvicorn.Server so uvicorn gets its own
            # clean event loop per thread without touching Jupyter's loop.
            # Do NOT call asyncio.set_event_loop() here — it conflicts with
            # asyncio.Runner inside uvicorn on Python 3.12.
            import asyncio
            cfg = uvicorn.Config(app, host=_host, port=_port, log_level="info")
            server = uvicorn.Server(cfg)
            asyncio.run(server.serve())


        _server_thread = threading.Thread(target=_run_server, daemon=True)
        _server_thread.start()


        # Wait (up to 45 s) until the port is actually bound before returning
        import socket as _socket
        _bound = False
        for _i in range(45):
            try:
                with _socket.create_connection(("127.0.0.1", _port), timeout=1):
                    _bound = True
                    break
            except OSError:
                time.sleep(1)

        if _bound:
            logger.info(f"... Server ready on http://127.0.0.1:{_port}  (background thread)")
        else:
            logger.error(
                f"!!! Server did NOT bind to port {_port} within 45 s.\n"
                "    Possible causes:\n"
                "    1. A previous uvicorn is still holding the port -> Runtime > Restart Runtime\n"
                "    2. An import error crashed the app -> check the traceback above\n"
                "    3. The port is blocked by Colab firewall\n"
                "    Re-run this cell after fixing the issue."
            )
    else:
        # Regular Python script: use blocking uvicorn.run()
        uvicorn.run(
            app,
            host=config.get("api.host", "0.0.0.0"),
            port=config.get("api.port", 8000),
            log_level="info"
        )


… Pydantic 2.13.4 loaded (V2=True)
… FastAPI loaded
… Loguru loaded
… Tenacity, NumPy, scikit-learn loaded


/tmp/ipykernel_22584/2380873528.py:105: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredPDFLoader
/tmp/ipykernel_22584/2380873528.py:106: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


… LangChain loaded
… Sentence-transformers loaded
… PyTesseract loaded
… pdf2image loaded
… OpenCV loaded


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
2026-05-24 16:50:31.427 | INFO     | __main__:_setup_external_tools:673 - … Tesseract: /usr/bin/tesseract
2026-05-24 16:50:31.477 | INFO     | __main__:<cell line: 0>:4023 - ======================================================================
2026-05-24 16:50:31.477 | INFO     | __main__:<cell line: 0>:4024 - ASISTEN BELAJAR SD v7.2
2026-05-24 16:50:31.479 | INFO     | __main__:<cell line: 0>:4025 - ======================================================================
2026-05-24 16:50:31.480 | INFO     | __main__:<cell line: 0>:4026 - 📁 Config          : confi

… RAGAS loaded successfully (v0.3.2)
… CUDA detected: 1 GPU(s)
   GPU Name       : Tesla T4
   CUDA Version   : 12.8
   Memory limit   : 80% of VRAM


INFO:     Started server process [22584]
INFO:     Waiting for application startup.
2026-05-24 16:50:33.541 | INFO     | __main__:lifespan:3179 - ============================================================
2026-05-24 16:50:33.542 | INFO     | __main__:lifespan:3180 - Startup: server akan menerima koneksi segera...
2026-05-24 16:50:33.543 | INFO     | __main__:lifespan:3181 -          Model loading berjalan di background thread.
2026-05-24 16:50:33.545 | INFO     | __main__:lifespan:3182 - ============================================================
2026-05-24 16:50:33.548 | INFO     | __main__:_init_assistant_background:3189 - [BG-INIT] Mulai load UltimateAssistantLCEL...
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token

## Cell 5

In [5]:



import yaml, os


os.chdir("/content/skripsi_backend")


os.environ["GURU_TOKEN"]  = "Guru2026"
os.environ["SISWA_TOKEN"] = "Siswa2026"
os.environ["CORS_ORIGINS"] = "*"


cfg_data = {
    "paths": {
        "vector_db":   "./vector_db_lcel",
        "evaluations": "./evaluations",
        "logs":        "./logs",
        "temp":        "./temp"
    },
    "llm": {
        "primary_model":      "deepseek-r1:7b",
        "llama_guard_model":  "llama-guard3:8b",
        "ollama_base_url":    "http://localhost:11434",
        "temperature":        {"qa": 0.1, "quiz": 0.4, "guard": 0.0},
        "timeout":            600,
        "guard_timeout":      600
    },
    "embedding": {
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "device":     "cuda"
    },
    "pdf": {
        "strategy":       "hi_res",
        "extract_images": False,
        "infer_tables":   False,
        "languages":      ["ind", "eng"],
        "ocr_languages":  "ind+eng",
        "ocr_fallback":   True,
        "ocr_min_chars_per_page": 50,
        "max_size_mb":    50
    },
    "retrieval": {
        "method":      "mmr",
        "top_k":       8,
        "fetch_k":     32,
        "lambda_mult": 0.8,
        "rerank":      {"enabled": True, "top_n": 5}
    },
    "relevance": {
        "threshold": 0.6,
        "methods":   ["cosine_similarity", "semantic_overlap"]
    },
    "quiz": {
        "default_questions":   3,
        "max_questions":       10,
        "chunks_per_question": 4,
        "max_retry":           3
    },
    "anti_hallucination": {
        "strict_mode":               True,
        "overlap_refuse_threshold":  0.20,
        "min_context_overlap":       0.20
    },
    "ragas": {
        "enabled":     True,
        "result_file": "./evaluations/ragas_results.jsonl"
    },
    "api": {
        "host":         "0.0.0.0",
        "port":         8000,
        "cors_origins": ["*"]
    },
    "auth": {
        "guru_token":  "Guru2026",
        "siswa_token": "Siswa2026"
    }
}


with open("config.yaml", "w") as f:
    yaml.dump(cfg_data, f, default_flow_style=False, allow_unicode=True)


!cat config.yaml

anti_hallucination:
  min_context_overlap: 0.2
  overlap_refuse_threshold: 0.2
  strict_mode: true
api:
  cors_origins:
  - '*'
  host: 0.0.0.0
  port: 8000
auth:
  guru_token: Guru2026
  siswa_token: Siswa2026
embedding:
  device: cuda
  model_name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
llm:
  guard_timeout: 600
  llama_guard_model: llama-guard3:8b
  ollama_base_url: http://localhost:11434
  primary_model: deepseek-r1:7b
  temperature:
    guard: 0.0
    qa: 0.1
    quiz: 0.4
  timeout: 600
paths:
  evaluations: ./evaluations
  logs: ./logs
  temp: ./temp
  vector_db: ./vector_db_lcel
pdf:
  extract_images: false
  infer_tables: false
  languages:
  - ind
  - eng
  max_size_mb: 50
  ocr_fallback: true
  ocr_languages: ind+eng
  ocr_min_chars_per_page: 50
  strategy: hi_res
quiz:
  chunks_per_question: 4
  default_questions: 3
  max_questions: 10
  max_retry: 3
ragas:
  enabled: true
  result_file: ./evaluations/ragas_results.jsonl
relevance:
  methods:
  - cosine

## Cell 6

In [6]:
NGROK_AUTH_TOKEN = "3Dqz8GFl7JT37YnNqlqN9GbY4Sl_5dkp4SNaRCG7ieBN7oWMP"

import subprocess, time, os
from pyngrok import ngrok, conf

if not NGROK_AUTH_TOKEN:
    raise ValueError("NGROK_AUTH_TOKEN kosong! Daftar di https://dashboard.ngrok.com/signup")

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill ALL ngrok processes at OS level to fully release the tunnel
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
print("... Killed all ngrok processes, waiting for edge release...")
time.sleep(10)  # Wait for ngrok edge to de-register the domain

# Wait for the backend server to be ready (poll port 8000 instead of blind sleep)
print("Menunggu backend siap di port 8000 (max 120 detik)...")
import socket as _ng_socket
_backend_ready = False
for _ng_i in range(120):
    try:
        with _ng_socket.create_connection(("127.0.0.1", 8000), timeout=1):
            _backend_ready = True
            print(f"... Backend ready setelah {_ng_i + 1}s")
            break
    except OSError:
        if _ng_i % 10 == 0 and _ng_i > 0:
            print(f"    masih menunggu... ({_ng_i}s)")
        time.sleep(1)

if not _backend_ready:
    raise RuntimeError(
        "Backend TIDAK berjalan di port 8000 setelah 120 detik!\n"
        "Kemungkinan solusi:\n"
        "1. Pastikan Cell 4 (Backend Code) sudah dijalankan\n"
        "2. Cek error di output Cell 4 (mungkin ada import error)\n"
        "3. Runtime > Restart Runtime lalu jalankan ulang Cell 4, lalu cell ini"
    )

# Connect ngrok tunnel with retry logic (edge may take time to release)
public_url = None
for attempt in range(5):
    try:
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        break
    except Exception as e:
        if attempt < 4:
            print(f"... Tunnel attempt {attempt+1} failed, retrying in 15s... ({e})")
            ngrok.kill()
            subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
            time.sleep(15)
        else:
            raise RuntimeError(
                f"Failed to start ngrok tunnel after 5 attempts: {e}\n"
                "Kemungkinan solusi:\n"
                "1. Buka https://dashboard.ngrok.com/tunnels → Stop All Tunnels\n"
                "2. Tunggu 2 menit lalu coba lagi\n"
                "3. Restart Colab runtime (Runtime → Restart Runtime)"
            )

print()
print("=" * 60)
print(f"BACKEND URL  : {public_url}")
print(f"API DOCS     : {public_url}/docs")
print(f"HEALTH CHECK : {public_url}/health")
print("=" * 60)
print()
print("Copy ke frontend .env.local:")
print(f"NEXT_PUBLIC_API_URL={public_url}")
print(f"NEXT_PUBLIC_GURU_TOKEN=Guru2026")
print(f"NEXT_PUBLIC_SISWA_TOKEN=Siswa2026")
print("=" * 60)

... Killed all ngrok processes, waiting for edge release...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Menunggu backend siap di port 8000 (max 120 detik)...
... Backend ready setelah 1s

BACKEND URL  : https://shampoo-navigate-result.ngrok-free.dev
API DOCS     : https://shampoo-navigate-result.ngrok-free.dev/docs
HEALTH CHECK : https://shampoo-navigate-result.ngrok-free.dev/health

Copy ke frontend .env.local:
NEXT_PUBLIC_API_URL=https://shampoo-navigate-result.ngrok-free.dev
NEXT_PUBLIC_GURU_TOKEN=Guru2026
NEXT_PUBLIC_SISWA_TOKEN=Siswa2026


## Cell 7 — Health check

In [7]:
# import requests, json

# r = requests.get("http://localhost:8000/health", timeout=10)
# print(json.dumps(r.json(), indent=2))

In [8]:
import subprocess, time, requests

# Kill proses ollama lama (kalau ada yang zombie)
!pkill -f "ollama serve" 2>/dev/null || true
time.sleep(2)

# Start fresh
subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=open("/tmp/ollama_err.log", "w"),
)

# Tunggu sampai Ollama benar-benar ready
for i in range(30):  # max 30 detik
    time.sleep(1)
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"✅ Ollama ready setelah {i+1} detik")
            break
    except:
        print(f"⏳ Menunggu... ({i+1}s)")
else:
    print("❌ Ollama gagal start. Cek: !cat /tmp/ollama_err.log")

!ollama list

^C


2026-05-24 16:50:47.043 | INFO     | __main__:__init__:1777 - LLMs initialized: deepseek-r1:7b


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ollama ready setelah 1 detik
NAME               ID              SIZE      MODIFIED      
llama-guard3:8b    46f211c3d866    4.9 GB    2 minutes ago    
deepseek-r1:7b     755ced02ce7b    4.7 GB    2 minutes ago    


In [9]:
import yaml, os

# Read current config
with open("/content/skripsi_backend/config.yaml") as f:
    cfg = yaml.safe_load(f)

# Update CORS untuk allow ngrok + localhost
cfg["api"]["cors_origins"] = [
    "http://localhost:3000",        # local frontend
    "https://*.ngrok-free.dev",     # any ngrok subdomain
    "https://*.ngrok.io",           # legacy ngrok
]

# Extend LLM timeout untuk slow generation
cfg["llm"]["timeout"] = 600
cfg["llm"]["temperature"]["qa"] = 0.3

# Save
with open("/content/skripsi_backend/config.yaml", "w") as f:
    yaml.dump(cfg, f)

print("✅ Config updated:")
!cat config.yaml | head -30

✅ Config updated:
anti_hallucination:
  min_context_overlap: 0.2
  overlap_refuse_threshold: 0.2
  strict_mode: true
api:
  cors_origins:
  - http://localhost:3000
  - https://*.ngrok-free.dev
  - https://*.ngrok.io
  host: 0.0.0.0
  port: 8000
auth:
  guru_token: Guru2026
  siswa_token: Siswa2026
embedding:
  device: cuda
  model_name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
llm:
  guard_timeout: 600
  llama_guard_model: llama-guard3:8b
  ollama_base_url: http://localhost:11434
  primary_model: deepseek-r1:7b
  temperature:
    guard: 0.0
    qa: 0.3
    quiz: 0.4
  timeout: 600
paths:
  evaluations: ./evaluations
  logs: ./logs


## Cell 8 — Upload PDF + test chat

## Cell 9 — Keep alive

In [ ]:
import time
i = 0
while True:
    i += 1
    print(f"\ralive {i * 30}s", end="", flush=True)
    time.sleep(30)

alive 30s

2026-05-24 16:50:51.921 | INFO     | __main__:__init__:1166 - … Cross-encoder reranker initialized
2026-05-24 16:50:52.093 | INFO     | __main__:__init__:1325 - … RAGASEvaluator: menggunakan HuggingFace embeddings
2026-05-24 16:50:52.094 | INFO     | __main__:__init__:1357 - … RAGASEvaluator ready (Ollama lokal — tanpa OpenAI)
2026-05-24 16:50:52.277 | INFO     | __main__:__init__:1054 - … LlamaGuardFilter siap — model: llama-guard3:8b
2026-05-24 16:50:52.282 | INFO     | __main__:_init_assistant_background:3192 - [BG-INIT] ✓ Assistant siap digunakan!


INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "OPTIONS /books HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /books HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /books HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /books HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /books HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
alive 60sINFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "OPTIONS /evaluate/ragas/summary HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /evaluate/ragas/summary HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "OPTIONS /evaluate/ragas HTTP/1.1" 200 OK


2026-05-24 16:52:17.813 | INFO     | __main__:evaluate_ragas:3726 - ✓ Pre-check: Ollama responding OK


INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK


...

2026-05-24 16:52:55.827 | INFO     | __main__:ask:2836 - Guard: {'passed': False, 'reason': 'out_of_context', 'overlap': 0.0, 'action': 'replaced'}
2026-05-24 16:52:55.936 | INFO     | __main__:ask:2881 - … Q&A | Score: 0.109 | Guard: replaced | Time: 38.10s
2026-05-24 16:52:55.937 | INFO     | __main__:_run_batch_sync:3759 - [RAGAS] Answer collection selesai, now evaluating...
2026-05-24 16:52:55.939 | INFO     | __main__:evaluate_batch:1443 - [RAGAS] evaluate_batch: start — 1 question(s), ground_truths=ada
2026-05-24 16:52:55.941 | INFO     | __main__:evaluate_batch:1446 - [RAGAS] evaluate_batch: membangun dataset...
2026-05-24 16:52:56.043 | INFO     | __main__:evaluate_batch:1449 - [RAGAS] evaluate_batch: menjalankan evaluasi RAGAS (ini bisa 2-10 menit)...
2026-05-24 16:52:56.044 | INFO     | __main__:_run_evaluate:1377 - [RAGAS] _run_evaluate: mulai — metrics=['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'], n_rows=1


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "OPTIONS /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
alive 180sINFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
alive 210sINFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GE

2026-05-24 16:54:56.208 | INFO     | __main__:_run_evaluate:1398 - [RAGAS] _run_evaluate: selesai dalam 120.2s
2026-05-24 16:54:56.254 | INFO     | __main__:evaluate_batch:1480 - [RAGAS] evaluate_batch: selesai (120.3s) — aggregate={'faithfulness': 0.0, 'answer_relevancy': 0.0, 'context_precision': 0.0, 'context_recall': 0.0}
2026-05-24 16:54:56.255 | INFO     | __main__:_run_batch_sync:3770 - [RAGAS] ✓ Batch evaluation selesai (158.4s)


INFO:     180.242.68.155:0 - "POST /evaluate/ragas HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /evaluate/ragas/summary HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
alive 300sINFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
alive 330sINFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     180.242.68.155:0 - "GET /status HTTP/1.1" 200 OK
INFO:     1

In [ ]:
# 1. Cek apakah GPU aktif
!nvidia-smi

# 2. Cek Ollama pakai GPU atau CPU
!cat /tmp/ollama_err.log | tail -20